In [8]:
# -*- coding: utf-8 -*-
# UI integrada: Aerogeofísica + SOM + Satélite (BDC/INPE/STAC)
# - Corrigido rasterio.sample (sem 'resampling')
# - Baixar itens STAC que intersectam a quadrícula -> sat_store[item_id]
# - Amostrar TCI/bandas (item único ou todos) para a grade interpolada

# ==== imports do seu projeto ====
from src import *                               # Build_mc, Upload_geof, pop_nodata, sintetic_grid, import_malha_cartog
from verde_source import regular, interp_at     # opcional (interp_at utilizado se presente)

# ==== libs ====
import os, re, json, types, importlib, warnings, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import Point, Polygon
from shapely.ops import transform as shp_transform

from tqdm import tqdm
import ipywidgets as W
from IPython.display import display, clear_output

from sklearn_som.som import SOM
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from pystac_client import Client
import rasterio
from rasterio.enums import Resampling
from pyproj import Transformer

warnings.filterwarnings("ignore")
%matplotlib inline

# ====================== ESTADO GLOBAL ======================
ESCALAS = ['25k','50k','100k','250k','1kk']

quadricula = {}         # {fid: { 'folha': Series(EPSG=...), 'gama_*': df, 'mag_*': df, 'geof_*': df, ... }}
data_grid = None        # nome da camada interpolada SOM (ex.: 'geof_1105_linear')
som_store = {}          # {k: {'som','imp','sca','feats','layer'}}
som_last_pred = None
bdc_items = []          # lista de pystac.Item da busca corrente
sat_store = {}          # { item_id: {'collection','datetime','bbox','assets':{name:path}, 'hrefs':{name:href}} }

# ====================== HELPERS GERAIS ======================
def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

def _scan_layers_from_quadricula(q):
    layers = set()
    for _, blob in (q or {}).items():
        for k, v in blob.items():
            if isinstance(v, pd.DataFrame):
                layers.add(k)
    return tuple(sorted(layers))

def _available_columns(q, layers):
    cols = set()
    for _, blob in (q or {}).items():
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                cols.update([c for c in df.columns if c not in ('X','Y','E_utm','N_utm')])
    cols = sorted(cols, key=lambda c: (c!='MDT', c))
    return tuple(cols)

def _global_min_max_numeric(q, ids, layers, column, remove_neg=False):
    vals = []
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                s = df[column]
                if pd.api.types.is_numeric_dtype(s):
                    if remove_neg: s = s[s >= 0]
                    if s.size: vals.append(s.to_numpy())
    if not vals: return None, None
    v = np.concatenate(vals)
    if v.size == 0 or np.all(np.isnan(v)): return None, None
    return float(np.nanmin(v)), float(np.nanmax(v))

def _plot_layers_for_column(q, ids, layers, column, remove_neg=False):
    plt.figure(figsize=(12,9)); ax = plt.gca()
    is_num = False
    for fid in ids:
        for lay in layers:
            df = q.get(fid, {}).get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                is_num = pd.api.types.is_numeric_dtype(df[column]); break
        if is_num: break
    if is_num:
        vmin, vmax = _global_min_max_numeric(q, ids, layers, column, remove_neg)
        if vmin is not None and vmax is not None and vmin == vmax: vmin, vmax = vmin-1e-9, vmax+1e-9
        norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin is not None else None
        cmap = cm.get_cmap('terrain')
    for fid in ids:
        for lay in layers:
            df = q.get(fid, {}).get(lay)
            if not isinstance(df, pd.DataFrame) or column not in df.columns: continue
            d = df if not (is_num and remove_neg) else df[df[column] >= 0]
            if d.empty: continue
            if is_num:
                ax.scatter(d.X.values, d.Y.values, c=d[column].values, s=0.1, cmap=cmap, norm=norm, marker='H')
            else:
                codes, _ = pd.factorize(d[column], sort=True)
                ax.scatter(d.X.values, d.Y.values, c=codes, s=0.1, cmap='tab20', marker='H')
    ax.set_aspect('equal'); ax.set_title(f'Pré-visualização • {column} • {len(ids)} folha(s) • {", ".join(layers)}')
    if is_num and norm is not None:
        cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        vmin, vmax = norm.vmin, norm.vmax; mid = (vmin+vmax)/2
        cbar.set_ticks([vmin, mid, vmax]); cbar.ax.set_yticklabels([f'{vmin:.3g}', f'{mid:.3g}', f'{vmax:.3g}'])
        cbar.set_label(f'{column} (min→máx)')
    plt.show()

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n: return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _infer_suffix_from_names(*names):
    for nm in names or []:
        m = re.search(r'(\d{4})', str(nm) if nm else '')
        if m: return m.group(1)
    return '0000'

def _norm_name(s): return re.sub(r'[^a-z0-9]+','',str(s).lower())

_SYNONYMS = {
    'GMT': {'gmt','magigrf','magr','igrf','mag','gmtigrf'},
    'MDT': {'mdt','alte','altura'},
    'CTCOR': {'ctcor','ctc','ct'},
    'eTh': {'eth','eth_ppm','thc','th_ppm','ethppm','th'},
    'eU': {'eu','uc','u','euppm','u_ppm'},
    'KPERC': {'kperc','kc','k','kpct','k_percent'},
    'UTHRAZAO': {'uthrazao','uratio','u_th','u/th','u_th_ratio'},
    'UKRAZAO': {'ukrazao','u_k','u/k','u_k_ratio'},
    'THKRAZAO': {'thkrazao','th_k','th/k','th_k_ratio'},
}

def _find_source_column(df, canonical):
    want = _norm_name(canonical)
    for c in df.columns:
        if _norm_name(c) == want: return c
    for s in _SYNONYMS.get(canonical, set()):
        for c in df.columns:
            if _norm_name(c) == s: return c
    return None

def _source_order_for_feature(canonical):
    return ['mag','gama'] if canonical in ('GMT','MDT') else ['gama','mag']

# ============ INTERPOLAÇÃO para a grade ============
def _interpolate_current_selection(quad, ids, gama_key, mag_key, features, psize, algo, noneg=False):
    suf = _infer_suffix_from_names(gama_key, mag_key); out_name = f"geof_{suf}_{algo}"
    for fid in ids:
        blob = quad.get(fid, {})
        gdf = blob.get(gama_key); mdf = blob.get(mag_key)
        if gdf is None and mdf is None: continue
        xu, yu = sintetic_grid(quad, fid, psize=int(psize))
        sources = {}
        if isinstance(gdf, pd.DataFrame):
            gsrc = gdf.copy()
            if noneg:
                for c in gsrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(gsrc[c]):
                        gsrc.loc[gsrc[c] < 0, c] = np.nan
            sources['gama'] = (np.asarray(gsrc['X']), np.asarray(gsrc['Y']), gsrc)
        if isinstance(mdf, pd.DataFrame):
            msrc = mdf.copy()
            if noneg:
                for c in msrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(msrc[c]):
                        msrc.loc[msrc[c] < 0, c] = np.nan
            sources['mag'] = (np.asarray(msrc['X']), np.asarray(msrc['Y']), msrc)
        if not sources: continue
        out = {'X': xu, 'Y': yu}
        for f in features:
            arr = None
            for src in _source_order_for_feature(f):
                if src not in sources: continue
                x, y, df = sources[src]
                col = _find_source_column(df, f)
                if col is None: continue
                arr = interp_at(x, y, df[col].to_numpy(), xu, yu, algorithm=algo, extrapolate=True)
                break
            if arr is None: arr = np.full_like(xu, np.nan, dtype='float32')
            out[f] = arr
        quad[fid][out_name] = pd.DataFrame(out)
    return out_name

# ============ helpers UI ============
def _rescan_from_quadricula():
    q = globals().get('quadricula', {})
    layers = _scan_layers_from_quadricula(q); w_layers.options = layers
    global data_grid
    pick = (data_grid,) if data_grid and data_grid in layers else (layers[:1] if layers else ())
    w_layers.value = pick if pick else ()
    cols = _available_columns(q, w_layers.value) or ('MDT',)
    w_cols.options = cols
    w_cols.value = tuple([c for c in ('MDT',) if c in cols]) or ((cols[0],) if cols else ())
    # SOM widgets dependentes
    numeric = []
    for _, blob in q.items():
        for lay in w_layers.value:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in cols:
                    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]): numeric.append(c)
    opts = sorted(set(numeric), key=lambda c: (c!='MDT', c)) or ['MDT']
    w_feats.options = opts
    keep = [c for c in w_feats.value if c in opts] or (['MDT'] if 'MDT' in opts else opts[:min(5,len(opts))])
    w_feats.value = tuple(keep)
    test_opts = []
    if data_grid:
        for fid, blob in q.items():
            if data_grid in blob and isinstance(blob[data_grid], pd.DataFrame):
                test_opts.append(fid)
    w_test_ids.options = tuple(sorted(test_opts))
    w_test_ids.value = tuple(sorted(test_opts))[:min(4, len(test_opts))]
    ks = sorted(list(som_store.keys()))
    w_k_apply.options = ks
    if ks: w_k_apply.value = ks[0]

def _normalize_xy(df):
    if not {'E_utm','N_utm'}.issubset(df.columns):
        if {'X','Y'}.issubset(df.columns): df = df.rename(columns={'X':'E_utm','Y':'N_utm'}).copy()
        else: raise ValueError("Camada sem 'X','Y' ou 'E_utm','N_utm'.")
    df = df.sort_values(['N_utm','E_utm'], ascending=[False, True], ignore_index=True, kind='mergesort')
    xs1d = np.sort(df['E_utm'].unique()); ys1d = np.sort(df['N_utm'].unique())
    nx, ny = xs1d.size, ys1d.size
    xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)
    return df, xs_mesh, ys_mesh, nx, ny

def _build_matrix_for_fids(quad, features, layer, fids=None):
    fids_all = sorted(quad.keys()) if fids is None else list(fids)
    all_blocks, slc, metas = [], {}, {}
    k = 0
    for fid in fids_all:
        blob = quad.get(fid, {})
        if layer not in blob: continue
        df = blob[layer].copy()
        try:
            df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        except Exception: continue
        metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}
        X = df[features].to_numpy(dtype='float32')
        if X.size == 0: continue
        all_blocks.append(X)
        slc[fid] = slice(k, k+len(X)); k += len(X)
    if not all_blocks: raise RuntimeError(f"Nenhuma folha com '{layer}' e as features escolhidas.")
    return np.vstack(all_blocks), slc, metas

def _qe(som, X_std):
    D = som.transform(X_std); return float(np.mean(np.min(D, axis=1)))

def _te_1d(som, X_std):
    D = som.transform(X_std)
    bmu = np.argmin(D, axis=1); D2 = D.copy(); D2[np.arange(D.shape[0]), bmu] = np.inf
    sbmu = np.argmin(D2, axis=1); return float(np.mean(np.abs(bmu - sbmu) > 1))

def _plot_classes(classes_by_fid, metas, n_clusters, flip_ns=False, titulo='Mapa preditivo (SOM)'):
    cmap = _make_discrete_cmap(n_clusters)
    bounds = np.arange(-0.5, n_clusters + 0.5, 1); norm = BoundaryNorm(bounds, ncolors=n_clusters, clip=True)
    fig, ax = plt.subplots(figsize=(10,10), facecolor='w')
    for fid in sorted(classes_by_fid.keys()):
        Z = classes_by_fid[fid];  Z = np.flipud(Z) if flip_ns else Z
        xs = metas[fid]['xs']; ys = metas[fid]['ys']
        ax.pcolormesh(xs, ys, Z, cmap=cmap, norm=norm, shading='nearest', rasterized=True)
    ax.set_aspect('equal'); ax.set_title(titulo)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=np.arange(n_clusters), pad=0.01)
    cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)]); cbar.set_label('Classes')
    plt.tight_layout(); plt.show()

def _predict_per_folha(som, X_std, slc, metas):
    out = {}
    for fid, s in slc.items():
        y = som.predict(X_std[s]); ny, nx = metas[fid]['ny'], metas[fid]['nx']
        out[fid] = y.reshape(ny, nx)
    return out

def som_build_long_table(quad, layer, classes_by_fid, metas, atributos, fids=None):
    rows = []; fids_iter = list(classes_by_fid.keys()) if fids is None else list(fids)
    for fid in fids_iter:
        if fid not in classes_by_fid: continue
        Z = classes_by_fid[fid]; blob = quad.get(fid, {})
        if layer not in blob: continue
        df = blob[layer].copy(); df, xs, ys, nx, ny = _normalize_xy(df)
        Zv = Z.ravel(order='C') if Z.shape==(ny,nx) else np.ravel(Z)[:ny*nx]
        cols_keep = [a for a in atributos if a in df.columns]
        sub = pd.DataFrame({'fid': fid, 'E_utm': df['E_utm'].to_numpy(), 'N_utm': df['N_utm'].to_numpy(), 'classe': Zv.astype(int)})
        for a in cols_keep: sub[a] = df[a].to_numpy()
        rows.append(sub)
    if not rows: raise RuntimeError("Sem dados para tabela longa.")
    return pd.concat(rows, axis=0, ignore_index=True)

def plot_boxplots_por_classe(
    df_long: pd.DataFrame,
    atributos: list[str],
    classes: list | None = None,
    ncols: int = 2,
    showfliers: bool = False,
    rotation: int = 45,
    sharey: bool = True,
    figsize_cell: tuple[float, float] = (4.2, 3.2),
    suptitle: str | None = None,
    orient: str = 'v',  # 'v' = caixas verticais (labels no eixo X) | 'h' = horizontais
):
    """
    Para cada classe, plota um boxplot com as DISTRIBUIÇÕES dos 'atributos'.
    - Subplot = 1 classe
    - Eixo X (orient='v'): nomes dos atributos (o que você pediu)
      Eixo Y = valores numéricos
    - Se orient='h': caixas horizontais com labels no eixo Y (opcional)
    """
    import math
    import numpy as np
    import matplotlib.pyplot as plt

    if classes is None:
        classes = sorted(pd.Series(df_long['classe']).dropna().unique())

    n_classes = len(classes)
    if n_classes == 0:
        print("[Boxplot] Nenhuma classe para plotar.")
        return None

    ncols = max(1, int(ncols))
    nrows = math.ceil(n_classes / ncols)

    fig_w = max(6.0, figsize_cell[0] * ncols)
    fig_h = max(3.0, figsize_cell[1] * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=(sharey and orient=='v'), sharex=(sharey and orient=='h'))
    axes = np.atleast_1d(axes).ravel()

    # Unifica limites se sharey/sharex
    if sharey and orient == 'v':
        all_vals = []
        for c in classes:
            sub = df_long[df_long['classe'] == c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size:
                        all_vals.append(s)
        y_min = np.nanmin(np.concatenate(all_vals)) if all_vals else None
        y_max = np.nanmax(np.concatenate(all_vals)) if all_vals else None
    else:
        y_min = y_max = None

    if sharey and orient == 'h':
        all_vals = []
        for c in classes:
            sub = df_long[df_long['classe'] == c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size:
                        all_vals.append(s)
        x_min = np.nanmin(np.concatenate(all_vals)) if all_vals else None
        x_max = np.nanmax(np.concatenate(all_vals)) if all_vals else None
    else:
        x_min = x_max = None

    for i, c in enumerate(classes):
        ax = axes[i]
        sub = df_long[df_long['classe'] == c]

        vals_list, labels = [], []
        for a in atributos:
            if a not in sub.columns:
                continue
            s = pd.to_numeric(sub[a], errors='coerce').dropna()
            if s.size:
                vals_list.append(s.values)
                labels.append(a)

        # rótulo humano da classe (começando em 1 se inteiro)
        try:
            c_label = int(c) + 1
        except Exception:
            c_label = c

        if not vals_list:
            ax.set_title(f"Classe {c_label} (sem dados)")
            ax.axis("off")
            continue

        if orient == 'v':
            ax.boxplot(vals_list, labels=labels, showfliers=showfliers)
            ax.set_xlabel("Atributo")
            ax.set_ylabel("Valor")
            ax.tick_params(axis='x', labelrotation=rotation)
            if y_min is not None and y_max is not None:
                pad = 0.03 * (y_max - y_min if y_max != y_min else 1.0)
                ax.set_ylim(y_min - pad, y_max + pad)
        else:  # horizontal
            ax.boxplot(vals_list, vert=False, labels=labels, showfliers=showfliers)
            ax.set_ylabel("Atributo")
            ax.set_xlabel("Valor")
            if x_min is not None and x_max is not None:
                pad = 0.03 * (x_max - x_min if x_max != x_min else 1.0)
                ax.set_xlim(x_min - pad, x_max + pad)

        ax.set_title(f"Classe {c_label}")

    # esconde eixos sobrando
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    if suptitle:
        fig.suptitle(suptitle)

    plt.tight_layout()
    plt.show()
    return fig


def plot_boxplots_por_atributo(df_long, atributos, classes=None, ncols=2, showfliers=False, rotation=45, sharey=True, figsize_cell=(4.0,3.2), suptitle=None):
    import math
    if classes is None: classes = sorted(pd.Series(df_long['classe']).dropna().unique())
    n = len(classes); ncols = max(1,int(ncols)); nrows = math.ceil(n/ncols)
    fig_w = max(6.0, figsize_cell[0]*ncols); fig_h = max(3.2, figsize_cell[1]*nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=sharey); axes = np.atleast_1d(axes).ravel()
    y_min = y_max = None
    if sharey:
        gvals=[]
        for c in classes:
            sub = df_long[df_long['classe']==c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size: gvals.append(s)
        if gvals:
            gcat = np.concatenate(gvals); y_min, y_max = np.nanmin(gcat), np.nanmax(gcat)
    for i,c in enumerate(classes):
        ax = axes[i]; sub = df_long[df_long['classe']==c]
        vals, labels = [], []
        for a in atributos:
            if a in sub.columns:
                s = pd.to_numeric(sub[a], errors='coerce').dropna()
                if s.size: vals.append(s.values); labels.append(a)
        lab = int(c)+1 if isinstance(c,(int,np.integer)) else c
        if not vals: ax.set_title(f"Classe {lab} (sem dados)"); ax.axis("off")
        else:
            ax.boxplot(vals, labels=labels, showfliers=showfliers); ax.set_title(f"Classe {lab}")
            ax.set_xlabel("Atributo"); ax.set_ylabel("Valor"); ax.tick_params(axis='x', labelrotation=rotation)
            if y_min is not None and y_max is not None:
                pad = 0.03*(y_max-y_min if y_max!=y_min else 1.0); ax.set_ylim(y_min-pad, y_max+pad)
    for j in range(i+1, len(axes)): axes[j].axis("off")
    if suptitle: fig.suptitle(suptitle)
    plt.tight_layout(); plt.show(); return fig

# ============================ WIDGETS BASE ============================
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YA', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')
w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')

w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')
w_load   = W.Button(description='Carregar brutos', button_style='success', icon='download')

w_feats_interp = W.SelectMultiple(
    options=['GMT','CTCOR','eTh','eU','KPERC','UTHRAZAO','UKRAZAO','THKRAZAO','MDT'],
    value=('GMT','CTCOR','eTh','eU','KPERC','MDT'),
    rows=8, description='Features (grid)'
)
w_psize  = W.IntSlider(min=50, max=1000, step=50, value=100, description='Pixel (m)')
w_algo   = W.Dropdown(options=[('Linear','linear'),('Cúbico','cubic')], value='linear', description='Algoritmo')
w_nonegI = W.Checkbox(value=False, description='Negativos→NaN (grid)')
w_interpolar = W.Button(description='Interpolar grade', icon='shuffle')

w_layers = W.SelectMultiple(options=(), rows=6, description='Camadas')
w_cols   = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=6, description='Colunas')
w_nonegP = W.Checkbox(value=False, description='Remover negativos (preview)')
w_refresh = W.Button(description='Atualizar', icon='refresh')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')

w_feats  = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=8, description='Features (SOM)')
w_sigma  = W.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.5, description='sigma')
w_iter   = W.IntSlider(min=500, max=30000, step=500, value=10000, description='max_iter')
w_seed   = W.IntSlider(min=0, max=9999, step=1, value=42, description='seed')
w_flip   = W.Checkbox(value=False, description='flip N-S no plot')
w_ks_train = W.SelectMultiple(options=tuple(range(3,31)), value=(8,12,16), rows=8, description='k p/ treinar')
w_train  = W.Button(description='Treinar SOM(s)', button_style='primary', icon='play')

w_test_ids = W.SelectMultiple(options=(), rows=8, description='Folhas (teste)')
w_seltest  = W.ToggleButton(value=False, description='Selecionar todas (teste)', icon='check')
w_k_apply  = W.Dropdown(options=[], description='k (aplicar)')
w_apply    = W.Button(description='Aplicar/Testar', icon='check-circle')
w_evalall  = W.Button(description='Comparar Ks (métricas)', icon='bar-chart')
w_clear_models = W.Button(description='Limpar modelos', icon='trash')

w_boxplots = W.Button(description='Boxplots por atributo', icon='bar-chart')
w_datagrid_label = W.HTML(value="<b>Camada SOM:</b> <i>—</i>")
w_models_label   = W.HTML(value="<b>Modelos treinados:</b> <i>—</i>")
w_out    = W.Output()

# ============================ CALLBACKS BASE ============================
def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids; w_selall.value = False

def on_selall_change(ch):
    if ch['name']=='value': w_ids.value = tuple(w_ids.options) if ch['new'] else ()

def on_seltest_change(ch):
    if ch['name']=='value': w_test_ids.value = tuple(w_test_ids.options) if ch['new'] else ()

def on_clear_clicked(_):
    w_filtro.value = ''; w_ids.value = ()

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        print('# Montando grade…')
        quad = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando dados brutos…')
        _g, _m = Upload_geof(quad, gama_xyz=w_gama.value, mag_xyz=w_mag.value, extend_size=int(w_ext.value))
        quad = pop_nodata(quad)
        globals()['quadricula'] = quad
        print(f'Folhas ativas: {len(quad)}')
        globals()['data_grid'] = None
        w_datagrid_label.value = "<b>Camada SOM:</b> <i>— (interpole primeiro)</i>"
        som_store.clear(); globals()['som_last_pred']=None; w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        print('Pronto. Dados brutos anexados. Agora execute a INTERPOLAÇÃO.')

def on_refresh_clicked(_):
    with w_out:
        clear_output()
        if 'quadricula' not in globals(): print('Carregue dados primeiro.'); return
        _rescan_from_quadricula(); print('Atualizado.')

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        if not w_layers.value: print('Nenhuma camada selecionada.'); return
        q = globals().get('quadricula', {})
        for col in w_cols.value:
            _plot_layers_for_column(q, w_ids.value, w_layers.value, col, remove_neg=w_nonegP.value)

def on_interpolar_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        feats_grid = list(w_feats_interp.value)
        if not feats_grid: print('Selecione ao menos 1 feature (grid).'); return
        q = globals().get('quadricula', {})
        if not q: print('Carregue dados brutos primeiro.'); return
        print(f"# Interpolando (algo={w_algo.value}, pixel={int(w_psize.value)} m)…")
        out_layer = _interpolate_current_selection(q, w_ids.value, w_gama.value, w_mag.value, feats_grid, int(w_psize.value), w_algo.value, w_nonegI.value)
        globals()['quadricula'] = q; globals()['data_grid'] = out_layer
        w_datagrid_label.value = f"<b>Camada SOM:</b> <code>{out_layer}</code>"
        print(f"→ Camada criada: {out_layer}")
        som_store.clear(); globals()['som_last_pred']=None; w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        if out_layer in w_layers.options: w_layers.value = (out_layer,)

def on_train_clicked(_):
    with w_out:
        clear_output()
        feats = list(w_feats.value)
        if not feats: print("Selecione ao menos 1 feature (SOM)."); return
        if not globals().get('data_grid'): print("Interpole a grade primeiro."); return
        layer = globals()['data_grid']; q = globals().get('quadricula', {})
        print(f"[TREINO] Montando matriz global de '{layer}'…")
        X_all, _, _ = _build_matrix_for_fids(q, feats, layer, fids=None)
        imp = SimpleImputer(strategy='median'); X_imp = imp.fit_transform(X_all)
        sca = StandardScaler().fit(X_imp); X_std = sca.transform(X_imp)
        ks = sorted(set(int(k) for k in w_ks_train.value)); 
        if not ks: print("Escolha ao menos um k."); return
        np.random.seed(int(w_seed.value))
        for k in ks:
            print(f" - SOM(k={k}, sigma={float(w_sigma.value)}, it={int(w_iter.value)})")
            som = SOM(m=int(k), n=1, sigma=float(w_sigma.value), dim=len(feats), max_iter=int(w_iter.value)); som.fit(X_std)
            som_store[k] = {'som': som, 'imp': imp, 'sca': sca, 'feats': feats, 'layer': layer}
        w_models_label.value = f"<b>Modelos treinados:</b> {', '.join(map(str, sorted(som_store.keys())))}"
        w_k_apply.options = sorted(list(som_store.keys()))
        if w_k_apply.options: w_k_apply.value = w_k_apply.options[0]
        print("Modelos treinados.")

def on_apply_clicked(_):
    with w_out:
        clear_output()
        if not som_store: print("Treine um SOM antes."); return
        if not w_test_ids.value: print("Selecione folhas para teste."); return
        k = int(w_k_apply.value); model = som_store.get(k)
        if model is None: print(f"k={k} não encontrado."); return
        feats = model['feats']; layer = model['layer']; q = globals().get('quadricula', {})
        print(f"[TESTE] Subset {len(w_test_ids.value)} folhas / layer '{layer}'…")
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e: print(str(e)); return
        X_te_std = model['sca'].transform(model['imp'].transform(X_te))
        qe = _qe(model['som'], X_te_std); te = _te_1d(model['som'], X_te_std)
        print(pd.DataFrame([{'k':k,'QE_test':qe,'TE_test':te}]).to_string(index=False))
        classes = _predict_per_folha(model['som'], X_te_std, slc_te, metas_te)
        _plot_classes(classes, metas_te, n_clusters=k, flip_ns=bool(w_flip.value),
                      titulo=f"SOM (aplicar) k={k} | sigma={float(w_sigma.value)} | it={int(w_iter.value)} | {layer}")
        globals()['som_last_pred'] = {'k':k, 'classes':classes, 'metas':metas_te, 'fids':tuple(w_test_ids.value), 'feats':tuple(feats), 'layer':layer}
        print("Predição salva: som_last_pred.")

def on_evalall_clicked(_):
    with w_out:
        clear_output()
        if not som_store or not w_test_ids.value: print("Treine/aplique SOM e selecione folhas."); return
        any_k = next(iter(som_store)); feats = som_store[any_k]['feats']; layer = som_store[any_k]['layer']
        q = globals().get('quadricula', {})
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e: print(str(e)); return
        rows=[]
        for k, model in sorted(som_store.items()):
            if model['feats']!=feats or model['layer']!=layer: rows.append({'k':k,'QE_test':np.nan,'TE_test':np.nan,'obs':'incompatível'}); continue
            X_te_std = model['sca'].transform(model['imp'].transform(X_te))
            rows.append({'k':k,'QE_test':_qe(model['som'],X_te_std),'TE_test':_te_1d(model['som'],X_te_std)})
        print(pd.DataFrame(rows).sort_values('QE_test', ascending=True, na_position='last').to_string(index=False))

def on_clear_models_clicked(_):
    som_store.clear(); globals()['som_last_pred']=None
    w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"; w_k_apply.options=[]
    with w_out: clear_output(); print("Modelos apagados.")


def on_boxplots_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Treine e aplique um SOM primeiro.")
            return
        lp = globals().get('som_last_pred')
        if lp is None:
            print("Nenhuma predição recente. Clique em 'Aplicar/Testar' e tente novamente.")
            return

        k      = lp['k']
        classes= lp['classes']
        metas  = lp['metas']
        fids   = lp['fids']
        feats  = list(lp['feats'])
        layer  = lp['layer']
        q      = globals().get('quadricula', {})

        try:
            df_long = som_build_long_table(q, layer, classes, metas, atributos=feats, fids=fids)
        except RuntimeError as e:
            print(str(e)); return

        print(f"[Boxplots] {len(fids)} folha(s), k={k}, layer='{layer}', atributos={feats}")
        # orient='v' => caixas verticais com labels no eixo X (o que você pediu)
        plot_boxplots_por_classe(df_long, atributos=feats, ncols=2, showfliers=False, orient='v', suptitle=f"SOM k={k}")



# liga
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_seltest.observe(on_seltest_change, names='value')
w_clear.on_click(on_clear_clicked)
w_load.on_click(on_load_clicked)
w_refresh.on_click(on_refresh_clicked)
w_plot.on_click(on_plot_clicked)
w_interpolar.on_click(on_interpolar_clicked)
w_train.on_click(on_train_clicked)
w_apply.on_click(on_apply_clicked)
w_evalall.on_click(on_evalall_clicked)
w_clear_models.on_click(on_clear_models_clicked)
w_boxplots.on_click(on_boxplots_clicked)
refresh_ids()

# ============================ BDC / STAC (INPE) ============================
BDC_ENDPOINT = "https://data.inpe.br/bdc/stac/v1"

def _aoi_bbox_from_ids(escala, ids):
    if not ids: return None
    gdf = import_malha_cartog(escala=escala)
    gdf = gdf[gdf['id_folha'].astype(str).isin([str(i) for i in ids])].copy()
    if gdf.empty: return None
    try: gdf = gdf.to_crs(4326)
    except Exception: pass
    minx,miny,maxx,maxy = gdf.total_bounds
    return [float(minx), float(miny), float(maxx), float(maxy)]

def _bdc_list_collections(pattern=None):
    cli = Client.open(BDC_ENDPOINT)
    cols = [c.id for c in cli.get_collections()]
    if pattern:
        pat = re.compile(pattern, re.IGNORECASE); cols = [c for c in cols if pat.search(c)]
    return sorted(cols)

# === BDC: listar bandas, resolver seleção e baixar todas por padrão ===
import os, re, requests
from collections import OrderedDict

def _bdc_list_item_bands(item, only_tiff=True):
    """
    Retorna uma lista de tuplas (key, href, media_type, roles_str, n_hint)
    filtrando para GeoTIFF por padrão ('.tif/.tiff' ou type contendo 'tiff').
    n_hint é apenas uma dica de número de bandas (None quando desconhecido).
    """
    rows = []
    for key, a in (item.assets or {}).items():
        href = getattr(a, "href", None) or (a.get("href") if isinstance(a, dict) else None)
        mtype = (getattr(a, "type", None) or getattr(a, "media_type", None) or "").lower()
        roles = getattr(a, "roles", None) or (getattr(a, "extra_fields", {},).get("roles") if hasattr(a, "extra_fields") else None)
        roles_str = ",".join(roles) if roles else ""

        if not href:
            continue
        if only_tiff:
            if not (href.lower().endswith((".tif", ".tiff")) or "tiff" in mtype):
                continue
        # heurística leve de multi-banda no nome:
        n_hint = 3 if key.lower() in ("tci", "visual") else (1 if re.search(r"\b(band|b)0?\d+\b", key.lower()) else None)
        rows.append((key, href, mtype, roles_str, n_hint))
    # ordena por chave
    rows.sort(key=lambda r: r[0].lower())
    return rows


def _bdc_print_item_bands(item, only_tiff=True, printer=print):
    """
    Imprime as bandas/ativos disponíveis (filtrando para GeoTIFF por padrão).
    """
    rows = _bdc_list_item_bands(item, only_tiff=only_tiff)
    if not rows:
        printer("Nenhuma banda GeoTIFF detectada neste item.")
        return rows
    printer("Bandas/ativos disponíveis (GeoTIFF):")
    for key, href, mtype, roles_str, n_hint in rows:
        base = os.path.basename(href)
        nh = f"{n_hint}b" if n_hint else "?"
        printer(f" - {key:>8} | {nh} | {mtype or 'type?':12} | {roles_str or '-':10} | {base}")
    return rows


def _bdc_download_item_bands(item, outdir="satellite_bdc", only_tiff=True, printer=print):
    """
    Baixa TODAS as bandas/ativos do item (GeoTIFF por padrão).
    Retorna lista de caminhos salvos.
    """
    os.makedirs(outdir, exist_ok=True)
    rows = _bdc_list_item_bands(item, only_tiff=only_tiff)
    if not rows:
        printer("Nada para baixar (sem GeoTIFF).")
        return []

    saved = []
    for key, href, mtype, roles_str, _ in rows:
        name = f"{item.collection_id}_{item.id}_{key}_{os.path.basename(href)}"
        fpath = os.path.join(outdir, name)
        if os.path.exists(fpath):
            printer(f"✓ já existe: {fpath}")
            saved.append(fpath)
            continue
        try:
            with requests.get(href, stream=True, timeout=90) as r:
                r.raise_for_status()
                with open(fpath, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk:
                            f.write(chunk)
            printer(f"↓ ok: {fpath}")
            saved.append(fpath)
        except Exception as e:
            printer(f"[WARN] Falha ao baixar {key}: {e}")
    return saved


def _resolve_band_assets(item, bands_text):
    """
    Converte a string de bandas em [(name, href, idxs)].
    NOVO:
      - Se bands_text ∈ {'', 'all', '*', 'todas'} ⇒ seleciona TODAS as bandas GeoTIFF do item.
      - Sempre imprime a lista de bandas detectadas antes de resolver.
    Regras especiais:
      - Para 'tci'/'visual' usa (1,2,3); demais ativos assumem (1,) por padrão.
      - Suporta apelidos red/green/blue ↔ B4/B3/B2 e B04/B03/B02.
      - Suporta padrão b8/band8/b08 etc.
    """
    # 1) mostra o que há
    _bdc_print_item_bands(item, only_tiff=True, printer=print)

    # 2) default = todas
    if not bands_text or str(bands_text).strip().lower() in {"all", "*", "todas"}:
        out = []
        for key, href, _mtype, _roles, n_hint in _bdc_list_item_bands(item, only_tiff=True):
            if key.lower() in ("tci", "visual"):
                out.append((key, href, (1, 2, 3)))
            else:
                # default 1 banda; se quiser, você pode abrir com rasterio para detectar count
                out.append((key, href, (1,)))
        print(f"[resolve] Selecionadas TODAS as bandas ({len(out)}).")
        return out

    # 3) seleção manual (igual à lógica anterior, mas com pequenos refinamentos)
    wanted = [b.strip() for b in str(bands_text).split(',') if b.strip()]
    out = []

    # mapa case-insensitive das chaves de asset
    assets_ci = {k.lower(): k for k in item.assets.keys()}

    def _pick(*keys):
        for k in keys:
            kk = assets_ci.get(k.lower())
            if kk:
                href = getattr(item.assets[kk], "href", None)
                if href:
                    return kk, href
        return None, None

    for w in wanted:
        lw = w.lower()

        if lw == 'tci':
            k, href = _pick('tci', 'visual')
            if not href:
                has_rgb = (
                    (assets_ci.get('b4') and assets_ci.get('b3') and assets_ci.get('b2')) or
                    (assets_ci.get('b04') and assets_ci.get('b03') and assets_ci.get('b02'))
                )
                if has_rgb:
                    raise RuntimeError("Item sem 'tci'/'visual'. Selecione B4,B3,B2 (ou B04,B03,B02).")
                raise RuntimeError("Item não oferece 'tci'/'visual'.")
            out.append(('tci', href, (1, 2, 3)))
            continue

        kk = assets_ci.get(lw)
        if kk:
            href = item.assets[kk].href
            out.append((kk, href, (1,)))
            continue

        if lw in ('red', 'b4', 'b04', 'band4'):
            k, href = _pick('B4', 'B04', 'red')
            if href:
                out.append(('red', href, (1,)))
                continue

        if lw in ('green', 'b3', 'b03', 'band3'):
            k, href = _pick('B3', 'B03', 'green')
            if href:
                out.append(('green', href, (1,)))
                continue

        if lw in ('blue', 'b2', 'b02', 'band2'):
            k, href = _pick('B2', 'B02', 'blue')
            if href:
                out.append(('blue', href, (1,)))
                continue

        m = re.fullmatch(r'b(?:and)?0?(\d+)', lw)
        if m:
            n = int(m.group(1))
            k, href = _pick(f'B{n}', f'B{n:02d}')
            if href:
                out.append((f'B{n}', href, (1,)))
                continue

        raise RuntimeError(f"Banda/asset '{w}' não encontrada.")

    print(f"[resolve] Selecionadas: {[n for n,_,_ in out]}")
    return out

    

def _bdc_search_items(collections, bbox, dt_range, cloud_min, cloud_max, limit, sort_dir):
    cli = Client.open(BDC_ENDPOINT)
    q = {"eo:cloud_cover": {"gte": int(cloud_min), "lte": int(cloud_max)}}
    sortby = ["properties.datetime"] if sort_dir == "asc" else ["-properties.datetime"]
    search = cli.search(collections=list(collections), bbox=bbox, datetime=dt_range, query=q, sortby=sortby, max_items=int(limit))
    return list(search.items())

def _bdc_pick_visual_asset(item):
    for key in ("tci","visual","overview","thumbnail"):
        a = item.assets.get(key)
        if a and a.href: return a.href, key
    for trip in (("B4","B3","B2"),("red","green","blue")):
        if all(k in item.assets for k in trip): return item.assets[trip[0]].href, trip[0]
    return None

def _bdc_preview_thumbs(items, max_show=12):
    n = min(len(items), max_show)
    if n == 0: print("Nenhum item."); return
    ncols = 4; nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 2.8*nrows)); axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        it = items[i]; ax = axes[i]; ax.axis("off")
        pair = _bdc_pick_visual_asset(it)
        title = f"{it.collection_id}\n{getattr(it,'datetime',None).date() if getattr(it,'datetime',None) else '—'}"
        ax.set_title(title, fontsize=9)
        if pair is None: ax.text(0.5,0.5,"sem preview",ha="center",va="center"); continue
        href, _ = pair
        try:
            r = requests.get(href, timeout=15); r.raise_for_status()
            from PIL import Image; from io import BytesIO
            img = Image.open(BytesIO(r.content)); ax.imshow(img)
        except Exception as e:
            ax.text(0.5,0.5,f"erro preview\n{e}",ha="center",va="center",fontsize=8)
    for j in range(i+1, len(axes)): axes[j].axis("off")
    plt.tight_layout(); plt.show()

# -------- Amostragem de bandas na grade --------
def _grid_epsg_from_blob(blob):
    v = blob.get('folha', None)
    if v is not None:
        for key in ('EPSG','epsg'):
            if hasattr(v, key): 
                try: return int(getattr(v, key))
                except Exception: pass
            if isinstance(v, dict) and key in v:
                try: return int(v[key])
                except Exception: pass
    if 'EPSG' in blob:
        try: return int(blob['EPSG'])
        except Exception: pass
    raise RuntimeError("Não foi possível inferir o EPSG da folha.")

def _open_remote_raster(href):
    try: return rasterio.open(href)
    except Exception: pass
    if not href.startswith('/vsicurl/'): return rasterio.open('/vsicurl/' + href)
    raise


def _resolve_band_assets(item, bands_text):
    """
    Converte a string de bandas em [(name, href, idxs)].
    Suporta:
      - 'tci'/'visual' (RGB, índices (1,2,3))
      - nomes exatos de assets do item (uma banda)
      - apelidos: red/green/blue -> B4/B3/B2 (ou B04/B03/B02)
      - padrões: B8, band8, B08 etc.
    """
    wanted = [b.strip() for b in str(bands_text).split(',') if b.strip()]
    out = []

    # mapa case-insensitive das chaves de asset
    assets_ci = {k.lower(): k for k in item.assets.keys()}

    def _pick(*keys):
        """Tenta retornar (asset_key_real, href) para a primeira key disponível."""
        for k in keys:
            kk = assets_ci.get(k.lower())
            if kk:
                href = getattr(item.assets[kk], "href", None)
                if href:
                    return kk, href
        return None, None

    for w in wanted:
        lw = w.lower()

        # 1) TCI / VISUAL (RGB)
        if lw == 'tci':
            k, href = _pick('tci', 'visual')
            if not href:
                has_rgb = (
                    (assets_ci.get('b4') and assets_ci.get('b3') and assets_ci.get('b2')) or
                    (assets_ci.get('b04') and assets_ci.get('b03') and assets_ci.get('b02'))
                )
                if has_rgb:
                    raise RuntimeError("Item sem 'tci'/'visual'. Selecione B4,B3,B2 (ou B04,B03,B02).")
                raise RuntimeError("Item não oferece 'tci'/'visual'.")
            out.append(('tci', href, (1, 2, 3)))
            continue

        # 2) nome exato do asset
        kk = assets_ci.get(lw)
        if kk:
            href = item.assets[kk].href
            out.append((kk, href, (1,)))
            continue

        # 3) apelidos RGB
        if lw in ('red', 'b4', 'b04', 'band4'):
            k, href = _pick('B4', 'B04', 'red')
            if href:
                out.append(('red', href, (1,)))
                continue

        if lw in ('green', 'b3', 'b03', 'band3'):
            k, href = _pick('B3', 'B03', 'green')
            if href:
                out.append(('green', href, (1,)))
                continue

        if lw in ('blue', 'b2', 'b02', 'band2'):
            k, href = _pick('B2', 'B02', 'blue')
            if href:
                out.append(('blue', href, (1,)))
                continue

        # 4) padrão genérico: B8, B08, band8, band08, etc.
        m = re.fullmatch(r'b(?:and)?0?(\d+)', lw)
        if m:
            n = int(m.group(1))
            k, href = _pick(f'B{n}', f'B{n:02d}')
            if href:
                out.append((f'B{n}', href, (1,)))
                continue

        # 5) não achou
        raise RuntimeError(f"Banda/asset '{w}' não encontrada.")

    return out

def _sample_asset_into_layer(quad, fids, layer_name, href, band_idxs=(1,), prefix='sat'):
    """
    Amostra 1+ bandas do 'href' sobre os pontos (X,Y) do DataFrame 'layer_name'.
    Usa sample() do rasterio (nearest). Colunas criadas: <prefix>, <prefix>_r/g/b, etc.
    """
    with _open_remote_raster(href) as ds:
        if ds.crs is None: raise RuntimeError("GeoTIFF sem CRS.")
        ok_cols = 0
        for fid in fids:
            blob = quad.get(fid, {})
            df = blob.get(layer_name)
            if not isinstance(df, pd.DataFrame) or not {'X','Y'}.issubset(df.columns): continue
            try: epsg_grid = _grid_epsg_from_blob(blob)
            except Exception as e: print(f" - {fid}: erro EPSG → {e}"); continue
            tr = Transformer.from_crs(f"EPSG:{epsg_grid}", ds.crs, always_xy=True)
            xx, yy = tr.transform(df['X'].to_numpy(), df['Y'].to_numpy())
            # amostragem em blocos
            def _batched(xa, ya, bs=200000):
                for i in range(0, xa.size, bs): yield xa[i:i+bs], ya[i:i+bs]
            for j, b in enumerate(band_idxs, 1):
                vals = np.full(df.shape[0], np.nan, dtype='float32'); k = 0
                try:
                    for xb, yb in _batched(xx, yy):
                        pts = list(zip(xb, yb))
                        # rasterio.sample não aceita 'resampling' -> nearest
                        it = ds.sample(pts, indexes=b)  # <--- FIX
                        out = np.fromiter((row[0] for row in it), dtype='float32', count=xb.size)
                        vals[k:k+xb.size] = out; k += xb.size
                except Exception as e:
                    print(f" - {fid}: erro amostrando banda {b} → {e}"); continue
                # nome de coluna
                if len(band_idxs)==3:
                    suffix = ('r','g','b')[j-1] if j<=3 else f'b{j}'
                    col = f"{prefix}_{suffix}"
                elif len(band_idxs)==1:
                    col = f"{prefix}"
                else:
                    col = f"{prefix}_b{b}"
                df[col] = vals.astype('float32', copy=False); ok_cols += 1
        return ok_cols

# -------- Download e cache local de assets STAC --------
def _download_assets_for_item(item, bands_text, outdir="satellite_bdc"):
    os.makedirs(outdir, exist_ok=True)
    bands = _resolve_band_assets(item, bands_text)
    assets_local = {}; hrefs = {}
    base_dir = os.path.join(outdir, f"{item.collection_id}_{item.id}")
    os.makedirs(base_dir, exist_ok=True)
    for name, href, _idxs in bands:
        fname = os.path.basename(href.split('?')[0])
        fpath = os.path.join(base_dir, f"{name}_{fname}")
        if not os.path.exists(fpath):
            with requests.get(href, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(fpath, "wb") as f:
                    for ch in r.iter_content(1<<20):
                        if ch: f.write(ch)
        assets_local[name] = fpath; hrefs[name] = href
    # guarda no sat_store
    sat_store[item.id] = {
        'collection': item.collection_id,
        'datetime': getattr(item, 'datetime', None),
        'bbox': getattr(item, 'bbox', None) or getattr(item, 'properties', {}).get('bbox'),
        'assets': assets_local,
        'hrefs': hrefs
    }
    return assets_local

# -------- UI BDC --------
w_bdc_filter = W.Text(placeholder='regex (ex.: landsat|sentinel|cbers)', description='Filtro')
w_bdc_list   = W.Button(description='Listar coleções', icon='list')
w_bdc_cols   = W.SelectMultiple(options=(), rows=8, description='Coleções')

w_bdc_date   = W.Text(value='2018-01-01/2025-12-31', description='Data (UTC)')
w_bdc_cloud  = W.IntRangeSlider(value=[0,100], min=0, max=100, step=1, description='Nuvens (%)')
w_bdc_limit  = W.IntSlider(value=20, min=1, max=200, step=1, description='Limite')
w_bdc_sort   = W.Dropdown(options=[('Mais antigo','asc'),('Mais recente','desc')], value='desc', description='Ordenar')

w_bdc_search = W.Button(description='Buscar itens', icon='search', button_style='info')
w_bdc_prev   = W.Button(description='Thumbnails', icon='image')
w_bdc_save   = W.Button(description='Baixar VISUAL', icon='download')

w_bdc_item    = W.Dropdown(options=(), description='Item', disabled=True)
w_bdc_bands   = W.Text(value='tci', description='Bandas')      # 'tci' | 'B4,B3,B2' | 'B8' | 'red,green,blue'
w_bdc_prefix  = W.Text(value='sat', description='Prefixo')
w_bdc_sample  = W.Button(description='Amostrar item', icon='plus-square', button_style='warning')

# novos botões p/ baixar / amostrar em lote
w_bdc_dl_all  = W.Button(description='Baixar itens (todos)', icon='download', button_style='success')
w_bdc_sm_all  = W.Button(description='Amostrar itens (todos)', icon='plus-square')
w_bdc_out     = W.Output()

def _format_item_label(it, i):
    coll = getattr(it, "collection_id", "") or ""
    dt   = getattr(it, "datetime", None)
    dts  = (dt.date().isoformat() if hasattr(dt, "date") else str(dt)) if dt else "—"
    props = getattr(it, "properties", {}) or {}
    cc = props.get("eo:cloud_cover") or props.get("cloud_cover")
    cc_str = (f"{cc:.0f}%" if isinstance(cc,(int,float)) else "—")
    return f"{i:02d} | {coll} | {dts} | clouds {cc_str}"

def on_bdc_list_clicked(_):
    with w_bdc_out:
        clear_output()
        try:
            cols = _bdc_list_collections(w_bdc_filter.value.strip() or None)
            if not cols: print("Nenhuma coleção encontrada.")
            else:
                w_bdc_cols.options = tuple(cols)
                print(f"{len(cols)} coleção(ões). Selecione e pesquise.")
        except Exception as e:
            print("Erro ao listar coleções:", e)

            
def on_bdc_search_clicked(_):
    with w_bdc_out:
        clear_output()
        if not w_bdc_cols.value:
            print("Selecione ao menos 1 coleção.")
            return
        bbox = _aoi_bbox_from_ids(w_escala.value, list(w_ids.value))
        if not bbox:
            print("Selecione folhas (à esquerda) para definirmos a área.")
            return
        print("AOI (bbox WGS84):", bbox)
        try:
            items = _bdc_search_items(
                collections=w_bdc_cols.value,
                bbox=bbox,
                dt_range=w_bdc_date.value.strip(),
                cloud_min=w_bdc_cloud.value[0],
                cloud_max=w_bdc_cloud.value[1],
                limit=int(w_bdc_limit.value),
                sort_dir=w_bdc_sort.value
            )
        except Exception as e:
            print("Erro na busca STAC:", e); return

        globals()['bdc_items'] = items
        # preenche o dropdown "Item"
        w_bdc_item.options = [
            (f"{i:02d} | {it.collection_id} | {getattr(it,'datetime',None).date() if getattr(it,'datetime',None) else '—'}",
             i)
            for i, it in enumerate(items)
        ]
        w_bdc_item.disabled = (len(items) == 0)
        if items:
            w_bdc_item.value = 0
            print(f"Encontrados {len(items)} item(ns). Use 'Listar bandas' ou 'Amostrar p/ grade'.")
        else:
            print("Nenhum item encontrado.")


def on_bdc_prev_clicked(_):
    with w_bdc_out: clear_output(); _bdc_preview_thumbs(globals().get('bdc_items', []), max_show=16)

def on_bdc_save_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Faça a busca primeiro."); return
        os.makedirs("satellite_bdc", exist_ok=True)
        saved=[]
        for it in items:
            pair = _bdc_pick_visual_asset(it)
            if not pair: continue
            href, key = pair
            name = os.path.basename(href.split('?')[0])
            fpath = os.path.join("satellite_bdc", f"{it.collection_id}_{it.id}_{key}_{name}")
            try:
                if not os.path.exists(fpath):
                    with requests.get(href, stream=True, timeout=60) as r:
                        r.raise_for_status()
                        with open(fpath,"wb") as f:
                            for ch in r.iter_content(1<<20):
                                if ch: f.write(ch)
                saved.append(fpath)
            except Exception as e:
                print(f"[WARN] Falha ao baixar {href}: {e}")
        if saved:
            print("Arquivos salvos:"); [print(" -",p) for p in saved]
        else:
            print("Nenhum asset visual pôde ser baixado.")

def on_bdc_sample_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Busque itens primeiro."); return
        idx = int(w_bdc_item.value)
        if not (0 <= idx < len(items)): print(f"Índice inválido 0..{len(items)-1}."); return
        if not globals().get('data_grid'): print("Interpole a grade (crie a camada SOM)."); return
        layer = globals()['data_grid']; q = globals().get('quadricula', {})
        fids_target = [fid for fid,blob in q.items() if layer in blob]
        item = items[idx]
        try:
            bands = _resolve_band_assets(item, w_bdc_bands.value)
        except Exception as e:
            print("Bandas:", str(e)); return
        print(f"Amostrando {[b[0] for b in bands]} → '{layer}' em {len(fids_target)} folha(s)…")
        total_cols=0
        for name, href, idxs in bands:
            try:
                cols = _sample_asset_into_layer(q, fids_target, layer_name=layer, href=href, band_idxs=tuple(idxs), prefix=(w_bdc_prefix.value or name))
                total_cols += cols; print(f"  - OK {name}: {cols} coluna(s).")
            except Exception as e:
                print(f"  - {name}: erro → {e}")
        if total_cols==0:
            print("Nenhuma coluna criada (verifique EPSG/GeoTIFF).")
        else:
            globals()['quadricula']=q; _rescan_from_quadricula(); print("Pronto. Novas colunas disponíveis no SOM.")

def on_bdc_dl_all_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Busque itens primeiro."); return
        bands_text = w_bdc_bands.value
        print(f"Baixando {len(items)} item(ns) ({bands_text})…")
        ok=0
        for it in items:
            try:
                local = _download_assets_for_item(it, bands_text, outdir="satellite_bdc")
                print(f" - {it.id}: {list(local.keys())}")
                ok += 1
            except Exception as e:
                print(f" - {it.id}: erro → {e}")
        print(f"Concluído. {ok}/{len(items)} item(ns) armazenados em sat_store.")

def on_bdc_sm_all_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Busque itens primeiro."); return
        if not globals().get('data_grid'): print("Interpole a grade (crie a camada SOM)."); return
        layer = globals()['data_grid']; q = globals().get('quadricula', {})
        fids_target = [fid for fid,blob in q.items() if layer in blob]
        bands_text = w_bdc_bands.value
        print(f"Amostrar TODOS os itens ({len(items)}), bandas={bands_text} → layer '{layer}' …")
        total_cols = 0; it_done = 0
        for it in items:
            try:
                bands = _resolve_band_assets(it, bands_text)
                for name, href, idxs in bands:
                    cols = _sample_asset_into_layer(q, fids_target, layer_name=layer, href=href, band_idxs=tuple(idxs), prefix=(w_bdc_prefix.value or name))
                    total_cols += cols
                it_done += 1
            except Exception as e:
                print(f" - {it.id}: erro → {e}")
        if total_cols==0:
            print("Nenhuma coluna criada (verifique EPSG/GeoTIFF).")
        else:
            globals()['quadricula']=q; _rescan_from_quadricula()
            print(f"OK. {it_done}/{len(items)} itens amostrados; {total_cols} coluna(s) adicionada(s).")
w_bdc_listbands = W.Button(description='Listar bandas', icon='list')
w_bdc_dlbands   = W.Button(description='Baixar bandas (item)', icon='download', button_style='success')

def on_bdc_listbands_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Faça a busca primeiro."); return
        idx = int(w_bdc_item.value)
        if not (0 <= idx < len(items)):
            print("Índice de item inválido."); return
        _bdc_print_item_bands(items[idx], only_tiff=True, printer=print)

def on_bdc_dlbands_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Faça a busca primeiro."); return
        idx = int(w_bdc_item.value)
        if not (0 <= idx < len(items)):
            print("Índice de item inválido."); return
        out = _bdc_download_item_bands(items[idx], outdir="satellite_bdc", only_tiff=True, printer=print)
        if out:
            print(f"{len(out)} arquivo(s) GeoTIFF salvo(s) em satellite_bdc/")
w_bdc_bands.value = ""   # ← vazio = TODAS por padrão

def on_bdc_sample_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Faça a busca primeiro (BDC → Buscar itens).")
            return
        idx = int(w_bdc_item.value)
        if not (0 <= idx < len(items)):
            print(f"Índice inválido. Escolha 0..{len(items)-1}.")
            return
        if not globals().get('data_grid'):
            print("Interpole a grade primeiro (crie a camada SOM).")
            return
        layer = globals()['data_grid']
        q = globals().get('quadricula', {})
        fids_target = [fid for fid, blob in q.items() if layer in blob]

        item = items[idx]
        try:
            # bands_text vazio => TODAS
            bands = _resolve_band_assets(item, w_bdc_bands.value)
        except Exception as e:
            print("Bandas:", str(e)); return

        total_cols = 0
        print(f"Amostrando {[b[0] for b in bands]} → layer '{layer}' em {len(fids_target)} folha(s)…")
        for name, href, idxs in bands:
            try:
                cols = _sample_asset_into_layer(
                    q, fids_target, layer_name=layer, href=href,
                    band_idxs=tuple(idxs), prefix=(w_bdc_prefix.value or name)
                )
                total_cols += cols
                print(f"  - OK {name}: {cols} coluna(s) adicionada(s).")
            except Exception as e:
                print(f"  - {name}: erro → {e}")

        if total_cols == 0:
            print("Nenhuma coluna foi criada (verifique EPSG/GeoTIFF).")
        else:
            globals()['quadricula'] = q
            _rescan_from_quadricula()
            print("Pronto. As novas colunas já podem ser usadas no SOM.")

# liga BDC
w_bdc_list.on_click(on_bdc_list_clicked)
w_bdc_search.on_click(on_bdc_search_clicked)
w_bdc_prev.on_click(on_bdc_prev_clicked)
w_bdc_save.on_click(on_bdc_save_clicked)
w_bdc_sample.on_click(on_bdc_sample_clicked)
w_bdc_dl_all.on_click(on_bdc_dl_all_clicked)
w_bdc_sm_all.on_click(on_bdc_sm_all_clicked)

# painel BDC
bdc_controls = W.VBox([
    W.HBox([w_bdc_filter, w_bdc_list]),
    W.HBox([w_bdc_cols]),
    W.HBox([w_bdc_date, w_bdc_cloud, w_bdc_limit, w_bdc_sort]),
    W.HBox([w_bdc_search, w_bdc_prev, w_bdc_save]),
    W.HBox([w_bdc_item, w_bdc_bands, w_bdc_prefix, w_bdc_sample]),
    W.HBox([w_bdc_dl_all, w_bdc_sm_all]),
    w_bdc_out
])
bdc_controls = W.VBox([
    W.HBox([w_bdc_filter, w_bdc_list]),
    W.HBox([w_bdc_cols]),
    W.HBox([w_bdc_date, w_bdc_cloud, w_bdc_limit, w_bdc_sort]),
    W.HBox([w_bdc_search, w_bdc_prev, w_bdc_save]),
    W.HBox([w_bdc_item, w_bdc_bands, w_bdc_prefix, w_bdc_sample]),
    W.HBox([w_bdc_listbands, w_bdc_dlbands]),
    w_bdc_out
])

w_bdc_listbands.on_click(on_bdc_listbands_clicked)
w_bdc_dlbands.on_click(on_bdc_dlbands_clicked)

# ============================ LAYOUT FINAL ============================
left = W.VBox([
    W.HBox([w_escala, w_filtro]),
    W.HBox([w_ids, W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_load, w_refresh, w_plot])]),
    W.HTML("<hr><b>Interpolação para grade</b>"),
    W.HBox([w_feats_interp, W.VBox([w_psize, w_algo, w_nonegI, w_interpolar])]),
    w_datagrid_label,
    W.HTML("<hr><b>Imagens de Satélite — BDC/INPE (STAC)</b>"),
    bdc_controls,
])

mid = W.VBox([
    W.HTML("<b>Pré-visualização</b>"),
    W.HBox([w_layers, w_cols]),
    w_nonegP,
    W.HTML("<hr><b>SOM — Treino</b>"),
    w_feats,
    W.HBox([w_sigma, w_iter, w_seed]),
    W.HBox([w_ks_train, w_train]),
    w_models_label
])

right = W.VBox([
    W.HTML("<b>SOM — Teste/Aplicação</b>"),
    W.HBox([w_test_ids, W.VBox([w_seltest, w_k_apply, w_apply, w_evalall, w_flip, w_clear_models, w_boxplots])])
])

ui = W.VBox([W.HBox([left, mid, right]), w_out])

def on_boxplots_attr_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Treine e aplique um SOM primeiro.")
            return
        lp = globals().get('som_last_pred')
        if lp is None:
            print("Nenhuma predição recente. Clique em 'Aplicar/Testar' e tente novamente.")
            return

        k      = lp['k']
        classes= lp['classes']
        metas  = lp['metas']
        fids   = lp['fids']
        feats  = list(lp['feats'])
        layer  = lp['layer']
        q      = globals().get('quadricula', {})

        try:
            df_long = som_build_long_table(q, layer, classes, metas, atributos=feats, fids=fids)
        except RuntimeError as e:
            print(str(e)); return

        print(f"[Boxplots por atributo] {len(fids)} folha(s), k={k}, layer='{layer}', atributos={feats}")
        # sharey=False => cada atributo com sua própria escala
        plot_boxplots_por_atributo(df_long, atributos=feats, ncols=2, showfliers=False,
                                   sharey=False, suptitle=f"SOM k={k} — por atributo")
w_boxplots_attr.on_click(on_boxplots_attr_clicked)
right = W.VBox([
    W.HTML("<b>SOM — Teste/Aplicação</b>"),
    W.HBox([w_test_ids, W.VBox([
        w_seltest, w_k_apply, w_apply, w_evalall, w_flip, w_clear_models,
        w_boxplots,            # já existente (por classe)
        w_boxplots_attr        # novo (por atributo)
    ])])
])

display(ui)


In [6]:
quadricula.keys()

dict_keys(['SF23_VC', 'SF23_VD', 'SF23_XC', 'SF23_ZA'])

In [ ]:
print(quadricula.keys())
print(quadricula['SC23_ZA_IV'].keys())
quadricula['SC23_ZA_IV']['geof_1089_linear'].describe().T

In [ ]:
# checagem rápida
req = ['som_model','som_X_std','som_slc','som_metas','som_feats','som_classes','som_layer']
missing = [r for r in req if r not in globals()]
if missing:
    raise RuntimeError(f"Rode o botão 'Rodar SOM' antes. Faltando: {missing}")

# montar a tabela longa e plotar boxplots por atributo (classes no eixo X)
attrs = list(som_feats)              # normalmente as mesmas usadas no SOM
layer = som_layer                    # p.ex. 'geof_1105_linear' ou 'geof_1105_cubic'
subset_fids = list(w_ids.value)      # opcional: só as folhas selecionadas

df_long = som_build_long_table(quadricula, layer, som_classes, som_metas, attrs, fids=subset_fids)
plot_boxplots_por_atributo(df_long, atributos=attrs, ncols=2)   # uma figura por atributo

# (opcional) inspeção "por classe": um painel de boxplots (um por atributo) para cada classe
# plot_boxplots_por_classe(df_long, classes=None, atributos=attrs, ncols=3)


In [ ]:
# dentro do on_run_clicked, depois de calcular:
classes_by_fid = som_predict_per_folha(som, X_std, slc, metas)

globals()['som_model']    = som
globals()['som_X_std']    = X_std
globals()['som_slc']      = slc
globals()['som_metas']    = metas
globals()['som_feats']    = feats
globals()['som_classes']  = classes_by_fid
globals()['som_layer']    = globals().get('data_grid')
print("[SOM] Estado salvo em variáveis globais: som_model, som_X_std, som_slc, som_metas, som_feats, som_classes, som_layer")


In [ ]:
# escolha (ou reutilize) a camada interpolada usada no SOM:
layer = globals()['data_grid']               # ex.: 'geof_1089_linear' ou 'geof_1105_cubic'

# atributos que você quer analisar; normalmente os mesmos do SOM:
attrs = list(globals().get('som_feats', [])) or ['GMT','CTCOR','eTh','eU','KPERC','MDT']

# opcional: restringir a um subconjunto de folhas (teste)
subset_fids = list(w_ids.value)  # por exemplo, as selecionadas na UI

# 1) tabela longa
df_long = som_build_long_table(quadricula, layer, som_classes, som_metas, attrs, fids=subset_fids)

# 2a) “30 listas de boxplots por atributo”: um gráfico por atributo, caixas = classes
plot_boxplots_por_atributo(df_long, atributos=attrs, ncols=2)

# 2b) (opcional) “um gráfico por classe”, caixas = atributos
# plot_boxplots_por_classe(df_long, classes=None, atributos=attrs, ncols=3)


In [ ]:
import re
import geopandas as gpd
import ipywidgets as W
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# --- util: lista de escalas disponíveis no seu geopackage ---
ESCALAS = ['25k','50k','100k','250k','1kk']

def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

# --- widgets ---
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YB', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')
w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')
w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')
w_load   = W.Button(description='Carregar dados', button_style='success', icon='download')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')
w_out    = W.Output()

def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids
    w_selall.value = False

def on_selall_change(change):
    if change['name'] == 'value':
        if change['new']:
            w_ids.value = tuple(w_ids.options)
        else:
            w_ids.value = ()

def on_clear_clicked(_):
    w_filtro.value = ''
    w_ids.value = ()

def plot_preview(ids):
    mc = import_malha_cartog(escala=w_escala.value, IDs=list(ids))
    plt.figure(figsize=(10,6))
    ax = mc.boundary.plot(color='k', linewidth=1)
    labels = mc.representative_point()
    for i, row in mc.iterrows():
        xy = labels.loc[i].xy
        ax.text(xy[0][0], xy[1][0], row['id_folha'], fontsize=7, ha='center')
    ax.set_aspect('equal')
    plt.title(f'{len(mc)} folha(s) – {w_escala.value}')
    plt.show()

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        plot_preview(w_ids.value)

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        print('# Montando grade…')
        quadricula = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando geofísica…')
        gdf_gama, gdf_mag = Upload_geof(quadricula, gama_xyz=w_gama.value, mag_xyz=w_mag.value, extend_size=int(w_ext.value))
        quadricula = pop_nodata(quadricula)
        print(f'Folhas ativas: {len(quadricula)}')
        # preview rápida do MDT da gama (se houver)
        try:
            plt.figure(figsize=(12,9))
            for fid in quadricula:
                if w_gama.value in quadricula[fid]:
                    df = quadricula[fid][w_gama.value]
                    plt.scatter(df.X, df.Y, c=df.MDT, s=0.1, cmap='terrain', marker='H')
            plt.axis('scaled'); plt.title('Preview MDT (gama)')
            plt.show()
        except Exception as e:
            print('Preview não disponível:', e)
        # guarda em variável global se você quiser reaproveitar
        globals()['quadricula'] = quadricula
        print('Pronto. Variável global `quadricula` atualizada.')

# ligações
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_clear.on_click(on_clear_clicked)
w_plot.on_click(on_plot_clicked)
w_load.on_click(on_load_clicked)

# inicializa
refresh_ids()

In [ ]:
display(
    W.VBox([
        W.HBox([w_escala, w_filtro]),
        W.HBox([w_ids, W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_plot, w_load])]),
        w_out
    ])
)

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import ipywidgets as W
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
# (adicione este import junto dos demais)
from matplotlib import cm, colors


# --- util: lista de escalas disponíveis no seu geopackage ---
ESCALAS = ['25k','50k','100k','250k','1kk']

def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

# ---------------- helpers novos ----------------
def _scan_layers_from_quadricula(q):
    """Varre todas as folhas da quadricula e retorna o conjunto de chaves que são DataFrames."""
    layers = set()
    for fid, blob in (q or {}).items():
        for k, v in blob.items():
            if isinstance(v, pd.DataFrame):
                layers.add(k)
    return tuple(sorted(layers))

def _available_columns(q, layers):
    """Lista colunas numéricas (e também categóricas) presentes nas camadas selecionadas."""
    cols = set()
    for fid, blob in (q or {}).items():
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in df.columns:
                    if c in ('X','Y'): 
                        continue
                    cols.add(c)
    # ordena com MDT em destaque se existir
    cols = sorted(cols, key=lambda c: (c!='MDT', c))
    return tuple(cols)


# (ATUALIZE) agora com colorbar linear para numéricos (min→máx)
def _plot_layers_for_column(q, ids, layers, column, remove_negatives=False):
    plt.figure(figsize=(12, 9))
    ax = plt.gca()
    printed_legend = False

    # checa se é numérica e, se for, calcula vmin/vmax globais
    is_numeric = False
    vmin = vmax = None
    # tenta detectar tipo pela primeira ocorrência
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                is_numeric = pd.api.types.is_numeric_dtype(df[column])
                break
        if is_numeric:
            break

    if is_numeric:
        vmin, vmax = _global_min_max_numeric(q, ids, layers, column, remove_negatives)
        if vmin is not None and vmax is not None and not np.isfinite([vmin, vmax]).all():
            vmin = vmax = None  # proteção
        if vmin is not None and vmax is not None and vmin == vmax:
            # evita faixa nula
            eps = 1e-9
            vmin, vmax = vmin - eps, vmax + eps
        norm = colors.Normalize(vmin=vmin, vmax=vmax)
        cmap = cm.get_cmap('terrain')

    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if not isinstance(df, pd.DataFrame) or column not in df.columns:
                continue
            d = df

            # filtro negativos para numérico
            if is_numeric and remove_negatives:
                d = d[d[column] >= 0]
                if d.empty:
                    continue

            if is_numeric:
                # usa escala global (norm) para linearidade min→máx
                sc = ax.scatter(d.X.values, d.Y.values, c=d[column].values,
                                s=0.1, cmap=cmap, norm=norm, marker='H')
            else:
                # categórico: legenda textual (mantido)
                codes, uniques = pd.factorize(d[column], sort=True)
                if not printed_legend:
                    print(f'Legenda (categórica) para {column}:')
                    for i, u in enumerate(uniques):
                        print(f'  {i} → {u}')
                    printed_legend = True
                sc = ax.scatter(d.X.values, d.Y.values, c=codes, s=0.1,
                                cmap='tab20', marker='H')

    ax.set_aspect('equal')
    ax.set_title(f'Pré-visualização • {column} • {len(ids)} folha(s) • camadas: {", ".join(layers)}')
    plt.axis('scaled')

    # adiciona a colorbar só para numérico
    if is_numeric and vmin is not None and vmax is not None:
        cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        mid = (vmin + vmax) / 2.0
        cbar.set_ticks([vmin, mid, vmax])
        cbar.ax.set_yticklabels([f'{vmin:.3g}', f'{mid:.3g}', f'{vmax:.3g}'])
        cbar.set_label(f'{column} (min→máx)', rotation=90)

    plt.show()

# ---------------- widgets ----------------
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YB', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')

w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')

w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')

# novos
w_layers = W.SelectMultiple(options=(), rows=6, description='Camadas')
w_cols   = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=6, description='Colunas')
w_noneg  = W.Checkbox(value=False, description='Remover negativos')
w_refresh = W.Button(description='Atualizar', icon='refresh')
w_load   = W.Button(description='Carregar dados', button_style='success', icon='download')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')
w_out    = W.Output()

# ---------------- callbacks ----------------
def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids
    w_selall.value = False

def on_selall_change(change):
    if change['name'] == 'value':
        w_ids.value = tuple(w_ids.options) if change['new'] else ()

def on_clear_clicked(_):
    w_filtro.value = ''
    w_ids.value = ()

def plot_preview(ids):
    mc = import_malha_cartog(escala=w_escala.value, IDs=list(ids))
    plt.figure(figsize=(10,6))
    ax = mc.boundary.plot(color='k', linewidth=1)
    labels = mc.representative_point()
    for i, row in mc.iterrows():
        xy = labels.loc[i].xy
        ax.text(xy[0][0], xy[1][0], row['id_folha'], fontsize=7, ha='center')
    ax.set_aspect('equal')
    plt.title(f'{len(mc)} folha(s) – {w_escala.value}')
    plt.show()

def _rescan_quadricula_and_widgets():
    """Atualiza w_layers e w_cols olhando a variável global `quadricula`."""
    q = globals().get('quadricula', {})
    layers = _scan_layers_from_quadricula(q)
    w_layers.options = layers
    # tenta selecionar automaticamente as camadas recém-carregadas (gama/mag) se existirem
    picked = [lay for lay in (w_gama.value, w_mag.value) if lay in layers]
    w_layers.value = tuple(picked) if picked else tuple(layers[:1]) if layers else ()
    # atualiza colunas disponíveis
    cols = _available_columns(q, w_layers.value)
    if not cols:
        cols = ('MDT',)
    w_cols.options = cols
    # mantém MDT se existir, senão primeira
    w_cols.value = tuple([c for c in ('MDT',) if c in cols]) or (cols[0],)

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        if not w_layers.value:
            print('Nenhuma camada selecionada. Carregue dados ou pressione "Atualizar".')
            return
        # plota uma figura por coluna selecionada
        q = globals().get('quadricula', {})
        for col in w_cols.value:
            _plot_layers_for_column(q, w_ids.value, w_layers.value, col, remove_negatives=w_noneg.value)

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        print('# Montando grade…')
        quad = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando geofísica…')
        gdf_gama, gdf_mag = Upload_geof(
            quad, 
            gama_xyz=w_gama.value, 
            mag_xyz=w_mag.value, 
            extend_size=int(w_ext.value)
        )
        quad = pop_nodata(quad)
        globals()['quadricula'] = quad  # atualiza variável global
        print(f'Folhas ativas: {len(quad)}')
        # atualiza camadas/colunas disponíveis a partir do conteúdo atual
        _rescan_quadricula_and_widgets()
        # preview rápida (se MDT existir)
        try:
            if 'MDT' in w_cols.options:
                print('Prévia rápida (MDT)…')
                _plot_layers_for_column(quad, w_ids.value, w_layers.value or (), 'MDT', remove_negatives=w_noneg.value)
        except Exception as e:
            print('Preview não disponível:', e)
        print('Pronto. Variável global `quadricula` atualizada.')

def on_refresh_clicked(_):
    with w_out:
        clear_output()
        if 'quadricula' not in globals():
            print('A variável global `quadricula` ainda não existe. Carregue dados primeiro.')
            return
        print('Re-escaneando `quadricula` para detectar novas camadas/colunas…')
        _rescan_quadricula_and_widgets()
        print('Atualizado. Use "Pré-visualizar" para ver.')

def on_layers_change(_):
    # quando camadas mudam, recalcula as colunas disponíveis
    q = globals().get('quadricula', {})
    cols = _available_columns(q, w_layers.value)
    if not cols:
        cols = ('MDT',)
    old_choice = set(w_cols.value)
    w_cols.options = cols
    # preserva seleção se possível
    keep = tuple([c for c in cols if c in old_choice]) or (cols[0],)
    w_cols.value = keep

# (NOVO) min/máx global para colunas numéricas
def _global_min_max_numeric(q, ids, layers, column, remove_negatives=False):
    vals = []
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                s = df[column]
                if pd.api.types.is_numeric_dtype(s):
                    if remove_negatives:
                        s = s[s >= 0]
                    if s.size:
                        vals.append(s.to_numpy())
    if not vals:
        return None, None
    v = np.concatenate(vals)
    if v.size == 0 or np.all(np.isnan(v)):
        return None, None
    return float(np.nanmin(v)), float(np.nanmax(v))



# ligações
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_clear.on_click(on_clear_clicked)
w_plot.on_click(on_plot_clicked)
w_load.on_click(on_load_clicked)
w_refresh.on_click(on_refresh_clicked)
w_layers.observe(on_layers_change, names='value')

# inicializa
refresh_ids()

# layout
ui = W.VBox([
    W.HBox([w_escala, w_filtro]),
    W.HBox([
        w_ids, 
        W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_load, w_refresh, w_plot]),
        W.VBox([w_layers, w_cols, w_noneg])
    ]),
    w_out
])

display(ui)

In [ ]:
quadricula.keys()

In [ ]:
quadricula

# Construindo Quadrícula

In [ ]:
list_geof = os.listdir('/home/database/geof/')
list_geof

In [ ]:
#quadricula = Build_mc(escala='1kk',ID=['SC23'],verbose=True)

## Adicionando dados brutos à Quadrícula

In [ ]:
#gama_1105,mag_1105=Upload_geof(quadricula,'gama_line_1105','mag_line_1105',600)
#gama_1039,mag_1039=Upload_geof(quadricula,'gama_1039','mag_1039',1100)
#gama_3022,mag_3022=Upload_geof(quadricula,'gama_3022','mag_3022',1100)
#gama_1089,mag_1089=Upload_geof(quadricula,'gama_line_1089','mag_line_1089',600)

In [ ]:
quadricula

In [ ]:
geof_list_ids = list(quadricula.keys())
print(len(geof_list_ids))
for id in geof_list_ids:
    print(f' - Folha: {id}')
    carta=quadricula[id]
    for data in list(carta.keys())[2:]:
        print(f'    - {data}')

In [ ]:
#quadricula=pop_nodata(quadricula)
len(quadricula.keys())

for id in list(quadricula.keys()):
    print(f' - Folha:  {id}')
    carta = quadricula[id]
    print(f'    - {list(carta.keys())}')

## Vizualisando Área de Estudo

In [ ]:
plt.figure()
for id in list(quadricula.keys()):
    carta=quadricula[id]
    print(id)
    plt.plot(*transform_to_carta_utm(carta['folha']).exterior.xy,color='black')
    for data in list(carta.keys())[2:]:
        if 'mag' in data:
            pass
        else:
            plt.scatter(carta[data].X,carta[data].Y,c=carta[data].MDT, cmap='terrain', s=0.1,marker='H')
            plt.axis('scaled')

            
plt.suptitle('Área de cobertura dos levantamentos aerogeofísicos')
plt.tight_layout()

## Visualizando dados Radiométricos Brutos

In [ ]:
#plot_raw_gama_data(gama_1105,suptitle='Dados Radiométricos brutos (Gama_1105.XYZ)')

In [ ]:
#plot_histograms(gama_1039)


#gama_1039.columns
#mag_1039.columns
#gama_1105.columns

## Removendo valores negativos das contagens radiométricas

In [ ]:
# print(f'{gama_1039.describe().T},    {gama_1105.describe().T}')

In [ ]:
gama_1089_positive = remove_negative_values(gama_1089)
#gama_1105_positive = remove_negative_values(gama_1105)
#plot_histograms(gama_1105_positive)
#plot_boxplots(gama_1105_positive,gama_FEAT)
#plot_raw_gama_data(gama_1105_positive,'Dados radiométricos tratadas : value <= 0 == 0.001')

In [ ]:
gama_1105_positive = remove_negative_values(gama_1105)
# Agora passando as features explicitamente:
plot_histograms(gama_1105_positive, cols=gama_FEAT)
plot_boxplots(gama_1105_positive, cols=gama_FEAT)

plot_raw_gama_data(gama_1105_positive, 'Dados radiométricos tratadas : value <= 0 == 0.001')

In [ ]:
plot_boxplots(gama_1105_positive, cols=gama_FEAT, per_feature=True, robust=True, q=(0.01, 0.99))

In [ ]:
gama_1039_positive=remove_negative_values(gama_1039,lista=['X','Y','LATITUDE','LONGITUDE','geometry'])
gama_1039_positive['UTHRAZAO']=gama_1039_positive['eU']/gama_1039_positive['eTh']
gama_1039_positive['UKRAZAO']=gama_1039_positive['eU']/gama_1039_positive['KPERC']
gama_1039_positive['THKRAZAO']=gama_1039_positive['eTh']/gama_1039_positive['KPERC']

plot_histograms(gama_1039_positive)
plot_raw_gama_data(gama_1039_positive,'Dados radiométricos tratadas : value <= 0 == 0.001')

# Interpolação dos dados Brutos

## CONSTRUINDO UM GRID SINTÉTICO

In [ ]:
# REMOVING PART OF THE SINTETIC GRID
'''
df_xu_yu = pd.DataFrame(np.array([xu,yu]))
df_xu_yu=df_xu_yu.T
df_xu_yu.rename(columns={0:'xu',1:'yu'},inplace=True)

df_xu_yu[(df_xu_yu.xu < 540937) & (df_xu_yu.yu > 8866937)]

df_xu_yu.drop(df_xu_yu[(df_xu_yu.xu < 540937) & (df_xu_yu.yu > 8866937)].index,inplace=True)
plt.figure(figsize=(18,12))

plt.scatter(df_xu_yu.xu,df_xu_yu.yu,s=0.1,marker='.')
plt.axis('scaled')
'''



## Método Cúbico

In [ ]:
# Test de área

# area=(344093.45426573796, 396417.36691108724, 7621768.799495494, 7677527.304557458)
# int((area[3]-area[2])/100),int((area[1]-area[0])/100)

In [ ]:
#traditional_interpolation(quadricula,'mag_3022','gama_3022','cubic','geof_3022')

In [ ]:
#list(quadricula['SF23_VC'].keys())

In [ ]:

#df = quadricula['SF23_VC']['geof_3022_cubic']
#plt.figure(figsize=(12,12))
#plt.scatter(x=df.X,y=df.Y,c=df.GMT,cmap='rainbow')
#plt.axis('scaled')

In [ ]:
# Print the output. a=

#descript_cubic = df.describe(percentiles)
#descript_cubic[['eU','eTh','KPERC','CTCOR','UTHRAZAO','THKRAZAO','UKRAZAO']].T

In [ ]:
#plot_histograms(geof_1089_cubic,suptitle='Distribuição dos dados radiométricos interpolados (cúbico, pixel 100m)')
#plot_raw_data(geof_1089_cubic,suptitle='Dados radiométricos interpolados (cúbico, pixel 100m)')

## Método Nearest

In [ ]:
#plot_histograms(geof_1089_nearest,suptitle='Distribuição dos dados radiométricos interpolados (nearest, pixel 100m)')

In [ ]:
#plot_raw_data(geof_1089_nearest,suptitle='Dados radiométricos interpolados (nearest, pixel 100m)')

## Método Linear

In [ ]:
#traditional_interpolation(quadricula,'mag_3022','gama_3022','linear','geof_3022')
#traditional_interpolation(quadricula,'mag_line_1105','gama_line_1105','linear','geof_1105')
traditional_interpolation(quadricula,'mag_line_1089','gama_line_1089','linear','geof_1089')

In [ ]:
plt.figure(figsize=(24,16))

# PLOTANDO A MALHA CARTOGRÁFICA
for id in list(quadricula.keys()):
    carta=quadricula[id]
    plt.plot(*transform_to_carta_utm(carta['folha']).exterior.xy,color='black')
    
    # PLOTANDO OS DADOS INTERPOLADOS
    for data in list(carta.keys())[2:]:
        # print(data)
        if 'geof' in data:
            plt.scatter(carta[data].X,carta[data].Y,c=carta[data].eU,cmap='rainbow',s=0.5,marker='H')
            plt.axis('scaled')
        # SE NÃO TIVER DADOS NÃO PLOTA NADA
        else:
            pass
        
plt.suptitle('Área de cobertura dos levantamentos aerogeofísicos')
plt.tight_layout()

In [ ]:
'''
plt.figure(figsize=(24,16))

for id in list(quadricula.keys()):
    carta=quadricula[id]
    plt.plot(*transform_to_carta_utm(carta['folha']).exterior.xy,color='black')
    
    for data in list(carta.keys())[2:]:
        if 'geof' in data:
            plt.scatter(carta[data].X,carta[data].Y,c=carta[data].MDT,cmap='terrain',s=0.5,marker='H')
            plt.axis('scaled')
        else:
            pass
        
plt.suptitle('Área de cobertura dos levantamentos aerogeofísicos')
plt.tight_layout()
'''

# Classificações Não-Supervisionadas

## Self-organizing maps (SOM)

In [ ]:
df = quadricula['SC23_ZA']['geof_1089_linear']
#plot_histograms(df,gama_FEAT)
#plot_boxplots(df,FEAT)
#plot_raw_gama_data(df,suptitle='Dados Radiométricos interpolados (Algoritmo: Linear)',figsize=(27,16))
#plot_raw_mag_data(df,suptitle='Dados Magnetométricos interpolados (Algoritmo: Linear)')

## Pixel size

In [ ]:
df_rs = df
df_rs.rename(columns={'X':'E_utm','Y':'N_utm'},inplace=True)
#df_rs.fillna(0,inplace=True)

xpixel_size = (df_rs.E_utm.max()-df_rs.E_utm.min())/df_rs.E_utm.unique().size
ypixel_size = (df_rs.N_utm.max()-df_rs.N_utm.min())/df_rs.N_utm.unique().size
print('x:', xpixel_size, 'y:', ypixel_size)

nx=df_rs.E_utm.unique().size
ny=df_rs.N_utm.unique().size
ratio=ny/nx
xs = df_rs.E_utm.values.reshape(ny, nx)
ys = df_rs.N_utm.values.reshape(ny, nx)

#print(gama_FEAT)
#plot_corr(df_rs[features], size=11, mask_upper=False, annot=True)
#plot_corr(df_rs[features], size=11, method='spearman', cluster=True)

features = ['GMT', 'CTCOR', 'eTh', 'eU', 'KPERC', 'UTHRAZAO', 'UKRAZAO', 'THKRAZAO', 'MDT']
plot_corr(df_rs[features], size=11)

data = StandardScaler().fit_transform(df[features].values)

# data = df_rs[features].values
n_clusters = 12
# NÚMERO DE CLASSES
lito_SOM = SOM(m=n_clusters,
               n=1,
           sigma=1.5,
             dim=len(features),
        max_iter=10000)

lito_SOM.fit(data)

# predição de classes
predictions = lito_SOM.predict(data)




# --- PESOS ---
W = np.asarray(lito_SOM.weights)  # (n_clusters, len(features))
absmax = np.nanmax(np.abs(W))
norm = TwoSlopeNorm(vmin=-absmax, vcenter=0.0, vmax=absmax)

fig, ax = plt.subplots(figsize=(10, 10), facecolor='w')
im = ax.imshow(W, cmap='RdBu_r', norm=norm, aspect='equal')  # imshow > matshow aqui

# valores dentro das células (opcional)
for (i, j), z in np.ndenumerate(W):
    ax.text(j, i, f'{z:0.2f}', ha='center', va='center', fontsize=8)

# labels corretos, no mesmo ax
ax.set_yticks(np.arange(n_clusters))
ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)], fontsize=9)
ax.set_xticks(np.arange(len(features)))
ax.set_xticklabels(features, rotation=55, fontsize=10, ha='left')

# pôr rótulos de X embaixo (imshow costuma deixar em cima)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)
ax.invert_yaxis()  # mantém "Classe 1" no topo visual

cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.08, label='weights')
cbar.set_ticks([-absmax, 0, absmax])

plt.tight_layout()
plt.show()


n_clusters=12

# reshape das classes (0..n_clusters-1)
classes = predictions.reshape(ny, nx)

# colormap discreto com N cores
base = matplotlib.colormaps.get_cmap('Set3')  # já vem com N cores
cmap = ListedColormap(base.colors[:n_clusters], name='Set3_N')

# fronteiras e norma para classes inteiras
bounds = np.arange(-0.5, n_clusters + 0.5, 1)
norm = BoundaryNorm(bounds, cmap.N, clip=True)

datafig, ax = plt.subplots(figsize=(12, 12), facecolor='w')
im = ax.pcolormesh(xs, ys, classes, cmap=cmap, norm=norm, shading='nearest', rasterized=True)

ax.set_xlim(xs.min(), xs.max())
ax.set_ylim(ys.min(), ys.max())
ax.set_aspect('equal')
ax.set_title('SOM - Aerogeophysical Data (SF23_YB_III4)')

# colorbar no MESMO fig
cbar = datafig.colorbar(im, ax=ax, orientation='horizontal', pad=0.02, ticks=np.arange(n_clusters))
cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)], fontsize=9)
cbar.set_label('Classes')

plt.tight_layout()
plt.show()


In [ ]:
#df = quadricula['SF23_YB_I1']['geof_1105_linear']
df_rs = df
df_rs.rename(columns={'X':'E_utm','Y':'N_utm'},inplace=True)

xpixel_size = (df_rs.E_utm.max()-df_rs.E_utm.min())/df_rs.E_utm.unique().size
ypixel_size = (df_rs.N_utm.max()-df_rs.N_utm.min())/df_rs.N_utm.unique().size
print('x:', xpixel_size, 'y:', ypixel_size)
nx=df_rs.E_utm.unique().size
ny=df_rs.N_utm.unique().size
ratio=ny/nx
xs = df_rs.E_utm.values.reshape(ny, nx)
ys = df_rs.N_utm.values.reshape(ny, nx)

features = ['GMT', 'CTCOR', 'eTh', 'eU', 'KPERC', 'UTHRAZAO', 'UKRAZAO', 'THKRAZAO', 'MDT']
plot_corr(df_rs[features], size=11)
data = StandardScaler().fit_transform(df[features].values)

# data = df_rs[features].values
n_clusters = 5
# NÚMERO DE CLASSES
lito_SOM = SOM(m=n_clusters,
               n=1,
           sigma=1.5,
             dim=len(features),
        max_iter=10000)

lito_SOM.fit(data)

# predição de classes
predictions = lito_SOM.predict(data)


# --- PESOS ---
W = np.asarray(lito_SOM.weights)  # (n_clusters, len(features))
absmax = np.nanmax(np.abs(W))
norm = TwoSlopeNorm(vmin=-absmax, vcenter=0.0, vmax=absmax)
fig, ax = plt.subplots(figsize=(10, 10), facecolor='w')
im = ax.imshow(W, cmap='RdBu_r', norm=norm, aspect='equal')  # imshow > matshow aqui

# valores dentro das células (opcional)
for (i, j), z in np.ndenumerate(W):
    ax.text(j, i, f'{z:0.2f}', ha='center', va='center', fontsize=8)
# labels corretos, no mesmo ax
ax.set_yticks(np.arange(n_clusters))
ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)], fontsize=9)
ax.set_xticks(np.arange(len(features)))
ax.set_xticklabels(features, rotation=55, fontsize=10, ha='left')
# pôr rótulos de X embaixo (imshow costuma deixar em cima)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)
ax.invert_yaxis()  # mantém "Classe 1" no topo visual
cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.08, label='weights')
cbar.set_ticks([-absmax, 0, absmax])
plt.tight_layout()
plt.show()

# reshape das classes (0..n_clusters-1)
classes = predictions.reshape(ny, nx)

# colormap discreto com N cores
base = matplotlib.colormaps.get_cmap('Set3')  # já vem com N cores
cmap = ListedColormap(base.colors[:n_clusters], name='Set3_N')

# fronteiras e norma para classes inteiras
bounds = np.arange(-0.5, n_clusters + 0.5, 1)
norm = BoundaryNorm(bounds, cmap.N, clip=True)

datafig, ax = plt.subplots(figsize=(12, 12), facecolor='w')
im = ax.pcolormesh(xs, ys, classes, cmap=cmap, norm=norm, shading='nearest', rasterized=True)

ax.set_xlim(xs.min(), xs.max())
ax.set_ylim(ys.min(), ys.max())
ax.set_aspect('equal')
ax.set_title('SOM - Aerogeophysical Data (SF23_YB_III4)')

# colorbar no MESMO fig
cbar = datafig.colorbar(im, ax=ax, orientation='vertical', pad=0.02, ticks=np.arange(n_clusters))
cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)], fontsize=9)
cbar.set_label('Classes')

plt.tight_layout()
plt.show()

In [ ]:
features = ['GMT','CTCOR', 'UTHRAZAO', 'UKRAZAO', 'THKRAZAO', 'MDT', 'eU', 'KPERC', 'eTh']
n_clusters=5
# ---------- 1) Agregar TODAS as quadrículas ----------
fids = sorted(quadricula.keys())
all_blocks, slc, metas = [], {}, {}
k = 0

data_grid = 'geof_1089_linear'

for fid in fids:
    if 'geof_1089_linear' not in quadricula[fid]:
        continue

    # 1) pega e garante UTM nas colunas
    df = quadricula[fid]['geof_1089_linear'].copy()
    if not {'E_utm','N_utm'}.issubset(df.columns):
        if {'X','Y'}.issubset(df.columns):
            df.rename(columns={'X': 'E_utm', 'Y': 'N_utm'}, inplace=True)
        else:
            raise KeyError(f"{fid}: não encontrei colunas E_utm/N_utm nem X/Y.")

    # 2) ordena para que o reshape (ny,nx) corresponda à malha
    df.sort_values(
        ['N_utm', 'E_utm'],
        ascending=[False,True],
        inplace=True,
        kind='mergesort',
        ignore_index=True
    )

    # 3) extrai malha a partir dos únicos (mais robusto que usar values.reshape)
    xs1d = np.sort(df['E_utm'].unique())
    ys1d = np.sort(df['N_utm'].unique())
    nx, ny = xs1d.size, ys1d.size
    xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)  # (ny,nx)

    # 4) guarda os metadados da folha
    metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}

    # 5) empilha as features (mesma ordem do sort acima!)
    X = df[features].to_numpy(dtype='float32')  # (ny*nx, nfeat)
    all_blocks.append(X)
    slc[fid] = slice(k, k + len(X))
    k += len(X)

X_all = np.vstack(all_blocks)  # (total_samples, nfeat)

# ---------- 2) Imputar e padronizar GLOBALMENTE ----------
# (mantém o shape por pixel; evita quebrar o reshape)
imp = SimpleImputer(strategy='median')
X_all_imp = imp.fit_transform(X_all)

scaler = StandardScaler().fit(X_all_imp)
X_all_std = scaler.transform(X_all_imp)

# ---------- 3) Treinar SOM ÚNICO ----------
som = SOM(m=n_clusters, n=1, sigma=1.5, dim=len(features), max_iter=3)
som.fit(X_all_std)

# (se quiser checar pesos)
W = np.asarray(som.weights)  # shape: (n_clusters, n_features)

# ---------- 4) Prever por quadrícula (classes consistentes) ----------
classes_by_fid = {}

for fid in fids:
    if data_grid not in quadricula[fid]:
        continue

    pr = som.predict(X_all_std[slc[fid]])  # vetor (N,)
    ny, nx = metas[fid]['ny'], metas[fid]['nx']
    classes_by_fid[fid] = pr.reshape(ny, nx)  # matriz (ny, nx)

epsg_default = 31983

for fid in sorted(classes_by_fid.keys()):
    xs = metas[fid]['xs']; ys = metas[fid]['ys']
    epsg = epsg_default  # ou epsg_por_fid.get(fid, epsg_default)

    # 1) salvar classes
    p1 = save_classes_tiff(fid, classes_by_fid[fid], xs, ys, epsg, outdir="out_som_classes_CTCOR")
    print("classes:", p1)

    # 2) (opcional) salvar stack de features
    if 'geof_1089_linear' in quadricula[fid]:
        df_interp = quadricula[fid]['geof_1089_linear']
        # garantir nomes UTM
        if {'E_utm','N_utm'}.issubset(df_interp.columns) is False and {'X','Y'}.issubset(df_interp.columns):
            df_interp = df_interp.rename(columns={'X':'E_utm','Y':'N_utm'})
        p2 = save_stack_tiff(fid, df_interp, features, xs, ys, epsg, outdir="out_geof_stack")
        print("stack:  ", p2)
        
plot_mapa_preditivo(classes_by_fid, metas, n_clusters=5, flip_ns=True)

In [ ]:
quadricula['SC23_ZA'].keys()

In [ ]:
run_btn.on_click(on_run_clicked)
ui = VBox([w_feats, HBox([w_k, w_sigma, w_iter]), HBox([w_seed, w_flip]), run_btn, out])
ui

In [ ]:
import pandas as pd

def ensure_geof_1105_linear(quadricula, feats, name='geof_1105_linear'):
    """
    Garante que cada folha tenha uma camada `name` com as colunas de `feats`.
    Monta a partir de quaisquer camadas que tenham ['X','Y'] e, ao menos, parte de `feats`.
    Se uma feature existir em várias camadas, usa o primeiro valor não-nulo.
    Retorna quantas folhas foram (re)criadas.
    """
    created = 0
    for fid, blob in (quadricula or {}).items():
        # se já existe e tem todas as feats, mantemos
        if isinstance(blob.get(name), pd.DataFrame) and set(feats).issubset(blob[name].columns):
            continue

        # coleta candidatos: dataframes com ['X','Y'] e alguma das feats
        candidates = []
        for lname, df in blob.items():
            if isinstance(df, pd.DataFrame) and {'X','Y'}.issubset(df.columns):
                pres = [f for f in feats if f in df.columns]
                if pres:
                    # só leva X,Y e as feats presentes
                    candidates.append((lname, df[['X','Y'] + pres].copy()))

        if not candidates:
            # nada para montar nesta folha
            continue

        # merge outer por X,Y acumulando colunas (com sufixo do layer para não colidir)
        base = None
        for lname, df in candidates:
            rename = {c: f"{c}__{lname}" for c in df.columns if c not in ('X','Y')}
            df2 = df.rename(columns=rename)
            base = df2 if base is None else pd.merge(base, df2, on=['X','Y'], how='outer')

        # reconstrói as colunas-alvo de feats escolhendo primeiro valor não-nulo entre as origens
        out = base[['X','Y']].copy()
        for f in feats:
            cols = [c for c in base.columns if c.startswith(f + '__')]
            if cols:
                out[f] = base[cols].bfill(axis=1).iloc[:, 0]
        # mantém apenas linhas com X,Y válidos
        out = out.dropna(subset=['X','Y'], how='any')

        # grava na folha
        blob[name] = out
        created += 1

    return created


In [ ]:
feats = quadricula['SC23_ZA']['geof_1089_linear'].keys()
print(feats)

feats = ['MDT', 'CTCOR', 'KPERC', 'eU', 'eTh', 'GMT', 'UTHRAZAO', 'UKRAZAO', 'THKRAZAO']

In [ ]:

# garante a camada que o build_global_matrix espera
made = ensure_geof_1105_linear(quadricula, feats, name='geof_1089_linear')
print(f"[info] geof_1105_linear (re)criadas em {made} folha(s).")

# agora sim, monta a matriz global
X_all, slc, metas = build_global_matrix(quadricula, feats)


In [ ]:
def debug_quadricula_columns(quadricula, max_folhas=3):
    c = 0
    for fid, blob in quadricula.items():
        print(f"\n== Folha {fid} ==")
        for lname, df in blob.items():
            if isinstance(df, pd.DataFrame):
                cols = [c for c in df.columns if c not in ('X','Y')]
                print(f"  - {lname}: {len(cols)} colunas (ex.: {cols[:15]})")
        c += 1
        if c >= max_folhas:
            break

debug_quadricula_columns(quadricula)


In [ ]:
import numpy as np
import streamlit as st
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# from seu_modulo import SOM
# pressupõe variável global `quadricula` carregada no Python que rodará o streamlit

st.set_page_config(page_title="SOM Geologia", layout="wide")
st.title("Mapa preditivo (SOM) – Aerogeofísica")

all_features = ['GMT','CTCOR','eTh','eU','KPERC','UTHRAZAO','UKRAZAO','THKRAZAO','MDT']
feats = st.multiselect("Features", all_features, default=all_features)
k = st.number_input("Número de classes (k)", min_value=2, max_value=50, value=12, step=1)
sigma = st.number_input("Sigma", min_value=0.1, max_value=5.0, value=1.5, step=0.1, format="%.1f")
max_iter = st.number_input("Iterações (max_iter)", min_value=500, max_value=30000, value=10000, step=500)
seed = st.number_input("Seed", min_value=0, max_value=9999, value=42, step=1)
flip_ns = st.checkbox("Inverter N-S no plot", value=False)

def make_discrete_cmap(n):
    if n <= 20:
        return matplotlib.cm.get_cmap('tab20', n)
    return matplotlib.cm.get_cmap('nipy_spectral', n)

def build_global_matrix(quadricula, features):
    fids = sorted(quadricula.keys())
    all_blocks, slc, metas = [], {}, {}
    k0 = 0
    for fid in fids:
        q = quadricula[fid]
        if 'geof_1105_linear' not in q: 
            continue
        df = q['geof_1105_linear'].copy()
        if not {'E_utm','N_utm'}.issubset(df.columns):
            if {'X','Y'}.issubset(df.columns):
                df.rename(columns={'X':'E_utm','Y':'N_utm'}, inplace=True)
            else:
                continue
        df.sort_values(['N_utm','E_utm'], ascending=[False, True], inplace=True, ignore_index=True, kind='mergesort')

        xs1d = np.sort(df['E_utm'].unique())
        ys1d = np.sort(df['N_utm'].unique())
        nx, ny = xs1d.size, ys1d.size
        xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)

        metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}
        X = df[features].to_numpy(dtype='float32')
        if X.size == 0: continue
        all_blocks.append(X)
        slc[fid] = slice(k0, k0 + len(X))
        k0 += len(X)
    if not all_blocks:
        raise RuntimeError("Nenhuma folha válida encontrada.")
    return np.vstack(all_blocks), slc, metas

def som_predict_per_folha(som, X_all_std, slc, metas):
    out = {}
    for fid, s in slc.items():
        pr = som.predict(X_all_std[s])
        ny, nx = metas[fid]['ny'], metas[fid]['nx']
        out[fid] = pr.reshape(ny, nx)
    return out

if st.button("Gerar mapa"):
    try:
        np.random.seed(int(seed))
        if len(feats) == 0:
            st.error("Selecione ao menos 1 feature.")
            st.stop()
        X_all, slc, metas = build_global_matrix(quadricula, feats)
        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X_all)
        scaler = StandardScaler().fit(X_imp)
        X_std = scaler.transform(X_imp)

        som = SOM(m=int(k), n=1, sigma=float(sigma), dim=len(feats), max_iter=int(max_iter))
        som.fit(X_std)

        classes_by_fid = som_predict_per_folha(som, X_std, slc, metas)

        cmap = make_discrete_cmap(int(k))
        bounds = np.arange(-0.5, int(k) + 0.5, 1)
        norm = BoundaryNorm(bounds, ncolors=int(k), clip=True)

        fig, ax = plt.subplots(figsize=(8,8), facecolor='w')
        for fid in sorted(classes_by_fid.keys()):
            Z = classes_by_fid[fid]
            if flip_ns:
                Z = np.flipud(Z)
            xs = metas[fid]['xs']; ys = metas[fid]['ys']
            ax.pcolormesh(xs, ys, Z, cmap=cmap, norm=norm, shading='nearest', rasterized=True)
        ax.set_aspect('equal')
        ax.set_title(f"SOM k={k} | sigma={sigma} | it={max_iter}")
        cbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=np.arange(int(k)), pad=0.01)
        cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(int(k))])
        cbar.set_label('Classes')
        st.pyplot(fig)

    except Exception as e:
        st.exception(e)


In [ ]:

# create labels
cluster_labels=[]
for i in range(n_clusters):
    cluster_labels+=[f'Classe {i+1}']
    
# classes weights
fig, ax = plt.subplots(figsize=(19,19))
im=ax.matshow(lito_SOM.weights)
for (i, j), z in np.ndenumerate(lito_SOM.weights):
    ax.text(j, i, '{:0.2f}'.format(z), ha='center', va='center')
    
plt.yticks(range(n_clusters), cluster_labels, fontsize=9)
plt.xticks(range(len(features)), features, rotation=55, fontsize=10, ha='left')
fig.colorbar(im, label='weights', orientation='horizontal')
plt.gca().set_aspect('equal')
plt.gcf().set_size_inches(10, 10)
plt.show()

In [ ]:
from matplotlib.colors import TwoSlopeNorm

# --- PESOS ---
W = np.asarray(lito_SOM.weights)  # (n_clusters, len(features))
absmax = np.nanmax(np.abs(W))
norm = TwoSlopeNorm(vmin=-absmax, vcenter=0.0, vmax=absmax)

fig, ax = plt.subplots(figsize=(10, 10), facecolor='w')
im = ax.imshow(W, cmap='RdBu_r', norm=norm, aspect='equal')  # imshow > matshow aqui

# valores dentro das células (opcional)
for (i, j), z in np.ndenumerate(W):
    ax.text(j, i, f'{z:0.2f}', ha='center', va='center', fontsize=8)

# labels corretos, no mesmo ax
ax.set_yticks(np.arange(n_clusters))
ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)], fontsize=9)
ax.set_xticks(np.arange(len(features)))
ax.set_xticklabels(features, rotation=55, fontsize=1a0, ha='left')

# pôr rótulos de X embaixo (imshow costuma deixar em cima)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)
ax.invert_yaxis()  # mantém "Classe 1" no topo visual

cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.08, label='weights')
cbar.set_ticks([-absmax, 0, absmax])

plt.tight_layout()
plt.show()


In [ ]:
id_ = [1,2,3,4,5,6,7,8,9]

relcolor =  matplotlib.cm.Set3
colors = np.array(relcolor.colors)[id_]
relcolor = matplotlib.colors.ListedColormap(colors)

idxs=np.arange(0, n_clusters, 1)
half=(idxs[1]-idxs[0])/2
ticks=np.linspace(idxs[0]+half, idxs[-1]-half, n_clusters)

datafig, ax=plt.subplots(figsize=(16, 16), facecolor='w')
im=plt.pcolormesh(xs, ys, predictions.reshape(ny, nx), cmap=relcolor, shading='auto')
plt.xlim(xs.min(), xs.max())
plt.ylim(ys.min(), ys.max())
cbar_ax = fig.add_axes([0.93, 0.3, 0.05, 0.4])
cbar = fig.colorbar(im, cax=cbar_ax, label = u'Classes', orientation='vertical',cmap='viridis',ticks=ticks)

cbar.ax.set_yticklabels(cluster_labels, fontsize=8)
plt.suptitle('SOM - Aerogeophysical Data (SF23_YA_III4)')
plt.axis('scaled')
plt.show()


In [ ]:
n_clusters = 11
lito_SOM = SOM(
    m=n_clusters,
    n=1,
    sigma=1.5,
    dim=len(features),
    max_iter=10000
)
lito_SOM.fit(data)

# predição de classes
predictions = lito_SOM.predict(data)

# create labels
cluster_labels=[]
for i in range(n_clusters):
    cluster_labels+=[f'Classe {i+1}']
    
# classes weights
fig, ax = plt.subplots(figsize=(27,27))
im=ax.matshow(lito_SOM.weights)

for (i, j), z in np.ndenumerate(lito_SOM.weights):
    ax.text(j, i, '{:0.2f}'.format(z), ha='center', va='center')

plt.yticks(range(n_clusters), cluster_labels, fontsize=8)
plt.xticks(range(len(features)), features, rotation=55, fontsize=9, ha='left')
fig.colorbar(im, label='weights', orientation='horizontal',cmap='Reds')
plt.gca().set_aspect('equal')
plt.gcf().set_size_inches(15, 10)
plt.show()

In [ ]:
from matplotlib.colors import TwoSlopeNorm
import numpy as np

W = np.asarray(lito_SOM.weights)  # (n_clusters, len(features))
absmax = np.nanmax(np.abs(W))
norm = TwoSlopeNorm(vmin=-absmax, vcenter=0.0, vmax=absmax)

fig, ax = plt.subplots(figsize=(15, 10))
im = ax.matshow(W, cmap='RdBu_r', norm=norm)

# valores nas células (opcional)
for (i, j), z in np.ndenumerate(W):
    ax.text(j, i, f'{z:0.2f}', ha='center', va='center')

# ---- LABELS NO EIXO CERTO (ax) ----
ax.set_yticks(np.arange(n_clusters))
ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)], fontsize=8)

ax.set_xticks(np.arange(len(features)))
ax.set_xticklabels(features, rotation=55, fontsize=9, ha='left')

# põe os rótulos de X embaixo (matshow costuma deixar em cima)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

# opcional: manter "Classe 1" no topo visual
ax.invert_yaxis()

# colorbar com 0 no meio
cbar = fig.colorbar(im, orientation='horizontal', label='weights', pad=0.08)
cbar.set_ticks([-absmax, 0, absmax])

fig.tight_layout()
plt.show()



In [ ]:
df

In [ ]:
id_ = [1,2,3,4,5,6,7,8,9,10,11]

relcolor =  matplotlib.cm.Set3
colors = np.array(relcolor.colors)[id_]
relcolor = matplotlib.colors.ListedColormap(colors)

idxs=np.arange(0, n_clusters, 1)
half=(idxs[1]-idxs[0])/2
ticks=np.linspace(idxs[0]+half, idxs[-1]-half, n_clusters)

datafig, ax=plt.subplots(figsize=(16, 16), facecolor='w')
im=plt.pcolormesh(xs, ys, predictions.reshape(ny, nx), cmap=relcolor, shading='auto')
plt.xlim(xs.min(), xs.max())
plt.ylim(ys.min(), ys.max())
cbar_ax = fig.add_axes([0.93, 0.3, 0.05, 0.4])
cbar = fig.colorbar(im, cax=cbar_ax, label = u'Classes', orientation='vertical',
                    ticks=ticks)
cbar.ax.set_yticklabels(cluster_labels, fontsize=8)
plt.suptitle('SOM - Aerogeophysical Data ()')
plt.axis('scaled')
plt.show()

## Testes

In [ ]:
df = quadricula['SF23_YA']['geof_1039_linear']
plot_histograms(df)
df.describe().T

#plot_raw_gama_data(df,suptitle='Dados Radiométricos interpolados (Algoritmo: Linear)',figsize=(27,16))
#plot_raw_mag_data(df,suptitle='Dados Magnetométricos i4nterpolados (Algoritmo: Linear)')

df = quadricula['SF23_YA']['geof_1105_linear']
plot_histograms(df)
df_rs = df

#df_rs.drop(columns=['CTCOR'],inplace=True)
#df_rs.fillna(0,inplace=True)
df_rs.rename(columns={'X':'E_utm','Y':'N_utm'},inplace=True)

xpixel_size = (df_rs.E_utm.max()-df_rs.E_utm.min())/df_rs.E_utm.unique().size
ypixel_size = (df_rs.N_utm.max()-df_rs.N_utm.min())/df_rs.N_utm.unique().size
print('x:', xpixel_size, 'y:', ypixel_size)


nx=df_rs.E_utm.unique().size
ny=df_rs.N_utm.unique().size
ratio=ny/nx
xs = df_rs.E_utm.values.reshape(ny, nx)
ys = df_rs.N_utm.values.reshape(ny, nx)

features = list(df_rs.columns[2:])
print(features)


plot_corr(df_rs[features], size=12)
#plt.savefig('figs/correlation_matrix.png', dpi=400, bbox_inches='tight')



scaler = StandardScaler()
data = scaler.fit_transform(df[features].values)
# data = df_rs[features].values


n_clusters = 11
lito_SOM = SOM(m=n_clusters, n=1, sigma=1.5, dim=len(features), max_iter=10)
lito_SOM.fit(data)

# predição de classes
predictions = lito_SOM.predict(data)

# create labels
#cluster_labels=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21]
cluster_labels=[1,2,3,4,5,6,7,8,9]

for i in range(n_clusters):
    cluster_labels+=[f'Classe {i+1}']
    
# classes weights
fig, ax = plt.subplots(figsize=(19,26))
im=ax.matshow(lito_SOM.weights)
for (i, j), z in np.ndenumerate(lito_SOM.weights):
    ax.text(j, i, '{:0.2f}'.format(z), ha='center', va='center')

plt.yticks(range(n_clusters), cluster_labels, fontsize=9)
plt.xticks(range(len(features)), features, rotation=55, fontsize=10, ha='left')
fig.colorbar(im, label='weights', orientation='horizontal')
plt.gca().set_aspect('equal')
plt.gcf().set_size_inches(19, 26)
plt.show()

id_ = [1,2,3,4,5,6,7,8,9]
relcolor =  matplotlib.cm.Set3
colors = np.array(relcolor.colors)[id_]
relcolor = matplotlib.colors.ListedColormap(colors)



idxs=np.arange(0, n_clusters, 1)
half=(idxs[1]-idxs[0])/2
ticks=np.linspace(idxs[0]+half, idxs[-1]-half, n_clusters)

datafig, ax=plt.subplots(figsize=(16, 16), facecolor='w')
im=plt.pcolormesh(xs, ys, predictions.reshape(ny, nx), cmap=relcolor, shading='auto')
plt.xlim(xs.min(), xs.max())
plt.ylim(ys.min(), ys.max())
cbar_ax = fig.add_axes([0.93, 0.3, 0.05, 0.4])
cbar = fig.colorbar(im, cax=cbar_ax, label = u'Classes', orientation='vertical',
                    ticks=ticks)
cbar.ax.set_yticklabels(cluster_labels, fontsize=8)
plt.suptitle('SOM - Aerogeophysical Data (SF23_YA_III4)')
plt.axis('scaled')
plt.show()

# Classificações Supervisionadas

## Rotulando amostras com classes litológicas

In [ ]:
import shapely.speedups
from shapely import geometry
shapely.speedups.enable()

geof_1089_linear['geometry'] = [geometry.Point(x,y) for x, y in zip(geof_1089_linear['X'], geof_1089_linear['Y'])]
gdf_1089_linear = geof_1089_linear.set_geometry('geometry')

gdf_1089_linear.set_crs('EPSG:32723',inplace=True)
gdf_1089_linear.geometry

In [ ]:
Upload_litologia(quadricula,'litologia_100k')

In [ ]:
litologia=quadricula['SB24_ZB_II']['litologia_100k']
litologia.to_crs('EPSG:32724',inplace=True)
print(litologia.crs)
litologia.reset_index(drop=True,inplace=True)

dic_litologico = describe_geologico(litologia)
print(litologia.columns)

In [ ]:
print(dic_litologico['SIGLA']['len'])
print(dic_litologico['SIGLA']['lista'])
gdf_1089_linear

In [ ]:
litologia.plot('SIGLA',figsize=(16,16),legend=True)

In [ ]:
geof_1089_linear['closest_unit'] = geof_1089_linear['geometry'].apply(lambda x: litologia['SIGLA'].iloc[litologia.distance(x).idxmin()]) # .idxmin() Retorna o indice do menor valor 

In [ ]:
geof_1089_linear.to_csv('/home/ggrl/database/csv/SB24_ZB_II_gama_linear_100m.csv',index=False)